# TPC Digital Current → primary-density analysis v24

This version keeps the synchronized DC input, exact mapping geometry, antenna handling, and v20-style readout recovery from v22, but changes the radial/phi analysis in the places that matter for the current data.

The key principle is: **do not make the data look smooth by hiding detector structure**. Instead, keep every intermediate detector-recovery stage and separately build a transparent smooth physics model.

### What is new in v24

- The empirical \(1/r^\alpha\) fit now uses the **actual 16 physical R1 rows**. In v22, `R1_FIT_N_LAYERS=12` accidentally selected 12 bins of the 0.1-cm display grid, which is only about two physical R1 rows.
- All recovery stages are retained: `measured → exposure → structure → stripes → outliers → smoothed`.
- Radial QA is built from the **48 physical TPC rows**, not repeated fine-grid cells.
- DC and gain-corrected primary profiles are plotted together. This directly tests whether R3 structure is already in DC or is introduced by the gain correction.
- A smooth two-source hypothesis is added:
  \[
  \rho(r,\phi)=A_{\rm coll}(R_0/r)^{1.23}
  +A_{\rm beam}(R_0/r)^2[1+\epsilon\cos(\phi-\phi_0)].
  \]
  The exponents are configurable. The collision term is flat in \(\phi\); the beam-background term has only the \(n=1\) harmonic by default.
- R2 and R3 have explicit response-scale nuisance parameters relative to R1. They are saved and plotted versus GEM current instead of being silently absorbed into the physics.
- The final smooth model, module-equalized recovered map, data/model ratio, and residual are all saved separately.
- The batch path is updated to the local SSD:
  `/media/yoren/T7_Shield/sPHENIX/tracking/dc/dc_run*.root`.
- All 57 early/middle/late files can be processed in parallel. **Run and window are both included in output names**, so the three windows of one run no longer overwrite each other.
- Cross-run output includes DC density vs GEM current, empirical R1 alpha, R2/R3 response scales, collision/beam amplitudes, beam \(n=1\) amplitude/direction, antenna ratios, and a static layer-response study.


### Additional v24 changes

- The two radial exponents are **fit parameters**, not fixed numbers. The collision-like component starts at 1.23 and is constrained to 1.0–1.5; the beam-background component starts at 2.0 and is constrained to 1.5–2.5. Their fitted values and approximate uncertainties are saved for QA.
- Antenna extraction now separates the **lookup φ convention of the DC global histogram** from the canonical output φ convention. The global DC histogram was filled with the preprocessing eBDC φ shift, while the final r–φ map uses the IdealPadMap-style canonical coordinates. v23 used the same φ for both, which can sample the wrong global bins.
- Both antenna lookup conventions are compared in QA.
- Raw antenna response is always preserved. An optional one-factor antenna response attachment can make the outer antenna ring continuous with the standard-R1 extrapolation without forcing the four antenna rings individually.


In [ ]:
import ROOT as root
import numpy as np
import pandas as pd
import math, os, re, subprocess, json, glob
from pathlib import Path
from array import array
from scipy import ndimage
from scipy.optimize import least_squares
from concurrent.futures import ThreadPoolExecutor, as_completed

root.gROOT.SetBatch(True)
root.TH1.AddDirectory(False)
root.gStyle.SetOptStat(0)

# ============================================================
# USER CONFIGURATION
# ============================================================

INPUT_ROOT = os.environ.get(
    "TPC_DC_INPUT",
    "input/dc0/dc_run00080858_win00.root"
)

MAPPING_DIR = Path(os.environ.get(
    "TPC_DC_MAPPING_DIR",
    "/home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/TPC/ChannelMapping1/PadPlane/"
))
GAIN_MAP_FILE = Path(os.environ.get(
    "TPC_DC_GAIN_MAP",
    "../distort/input/layer_gain_79513_Mariia_side01.root"
))

GAIN_MAP_HIST_SIDE0 = "hGainMap_side0_South"
GAIN_MAP_HIST_SIDE1 = "hGainMap_side1_North"
GAIN_LAYER_OFFSET = 7
APPLY_STANDARD_PAD_GAIN = True

DC_VARIANTS = ("raw", "ped60_signed", "ped60_positive", "ped60_strict")
MAIN_VARIANT = os.environ.get("TPC_DC_VARIANT", "ped60_signed")
DEFECT_VARIANT = os.environ.get("TPC_DC_DEFECT_VARIANT", "ped60_positive")
if MAIN_VARIANT not in DC_VARIANTS:
    raise ValueError(f"Unknown MAIN_VARIANT={MAIN_VARIANT}")
if DEFECT_VARIANT not in DC_VARIANTS:
    raise ValueError(f"Unknown DEFECT_VARIANT={DEFECT_VARIANT}")

N_SIDES, N_SECTORS, N_MODULES, LAYERS_PER_MODULE = 2, 12, 3, 16
N_PADS = {0:94, 1:128, 2:192}
PHI_MIN, PHI_MAX = -math.pi, math.pi
GLOBAL_PHI_BINS = 720
SOURCE_R_MIN_CM = 22.0
SOURCE_R_MAX_CM = 76.0
GLOBAL_DR_CM = 0.10

# Keep PHGarfield source range separately.
PHGARFIELD_R_MIN_CM = 22.8
PHGARFIELD_R_MAX_CM = 75.43

# ------------------------------------------------------------
# Native defect detection / recovery, inherited from v20
# ------------------------------------------------------------
MIN_EXPOSURE_FRACTION = 0.50
DEAD_CELL_REL_THRESHOLD = 0.18

MIN_DEAD_FEE_WIDTH = 2
DEAD_FEE_LAYER_FRACTION = 0.45
FEE_BRIDGE_PADS = 2
FEE_CONTEXT_PADS = 12
FEE_MIN_CONTEXT_CELLS = 4
FEE_CONTEXT_MEDIAN_LAYERS = 3
FEE_CONTEXT_GAUSS_SIGMA_LAYERS = 0.8

STRIPE_REL_THRESHOLD = 0.28
FULL_STRIPE_FRACTION = 0.70
HALF_STRIPE_FRACTION = 0.70
OTHER_HALF_MAX_DEAD_FRACTION = 0.35
STRIPE_NEIGHBOR_LAYERS = 3
ALLOW_R3_HALF_STRIPES = True

MIN_DEAD_BLOCK_WIDTH = 2
MIN_DEAD_BLOCK_LAYERS = 2
MIN_DEAD_BLOCK_CELLS = 6
BLOCK_CONTEXT_PADS = 12
BLOCK_CONTEXT_LAYERS = 3

REPAIR_SINGLE_OUTLIERS = True
SINGLE_DEAD_FACTOR = 0.25
SINGLE_HOT_FACTOR = 2.5
SINGLE_OUTLIER_PAD_RADIUS = 2
SINGLE_OUTLIER_LAYER_RADIUS = 1
SINGLE_OUTLIER_MIN_NEIGHBORS = 5
SINGLE_OUTLIER_PASSES = 2

# Native smoothing is intentionally modest. Strong smoothing is done later
# by the explicit physics model, and all intermediate stages are retained.
SMOOTH_NATIVE_MAPS = True
SMOOTH_PAD_RADIUS = 2
SMOOTH_LAYER_RADIUS = 1
SMOOTH_BLEND = 0.75

ADC_ESTIMATOR = "trimmed_mean"
TRIM_FRACTION = 0.15
CLIP_OUTLIERS = True
CLIP_NSIGMA = 5.0

# ------------------------------------------------------------
# Empirical R1 power-law fits
# ------------------------------------------------------------
ALPHA_BOUNDS = (0.0, 4.0)
ALPHA_INITIAL = 1.8
R1_FIT_N_LAYERS = 12       # now means 12 PHYSICAL R1 layers, not 0.1-cm grid bins
R1_FIT_RMIN_CM = 31.0
R1_FIT_RMAX_CM = 40.5

# ------------------------------------------------------------
# Smooth two-source physics model
#
# rho(r,phi) =
#   A_collision * (Rref/r)^alpha_collision
# + A_beam      * (Rref/r)^alpha_beam
#     * [1 + epsilon*cos(phi-phi0)]
#
# The two alpha values are kept configurable. By default they encode the
# expected broad collision component (~1.23, flat phi) and a more central
# beam-background component (~2, one preferred phi direction).
# ------------------------------------------------------------
COLLISION_ALPHA_INITIAL = float(os.environ.get("TPC_DC_COLLISION_ALPHA_INITIAL", "1.23"))
COLLISION_ALPHA_BOUNDS = (1.0, 1.5)
BEAM_ALPHA_INITIAL = float(os.environ.get("TPC_DC_BEAM_ALPHA_INITIAL", "2.0"))
BEAM_ALPHA_BOUNDS = (1.5, 2.5)
PHYSICS_MODEL_RREF_CM = 40.0
PHYSICS_MODEL_RMIN_CM = 31.0
PHYSICS_MODEL_RMAX_CM = 76.0
PHYSICS_MODEL_PHI_BINS = 72
PHYSICS_MODEL_MIN_PER_BIN = 6
PHYSICS_MODEL_EPS_MAX = 0.80
PHYSICS_MODEL_SCALE_MIN = 0.50
PHYSICS_MODEL_SCALE_MAX = 2.00
PHYSICS_MODEL_ERROR_FLOOR = 0.03

# Layer-response calibration is derived across all files as a separate QA
# product. Do not silently apply it during the first pass.
APPLY_LAYER_RESPONSE_CORRECTION = (
    os.environ.get("TPC_DC_APPLY_LAYER_CORRECTION", "0") == "1"
)
LAYER_RESPONSE_FILE = Path(os.environ.get(
    "TPC_DC_LAYER_RESPONSE_FILE",
    "output_dc/TPC_DC_layer_response_v24.csv"
))

# ------------------------------------------------------------
# Antenna QA
# ------------------------------------------------------------
ANTENNA_LOG_MAD_NSIGMA = 6.0
ANTENNA_MIN_EXPOSURE_FRACTION = 0.50
ANTENNA_REPAIR_PHI_RADIUS = 2
ANTENNA_REPAIR_RING_RADIUS = 1
ANTENNA_REPAIR_PASSES = 2

# The global DC histogram was filled in the preprocessing eBDC phi convention.
# Keep that lookup coordinate separate from the canonical IdealPadMap-like phi
# used to place the reconstructed antenna cell into the final r-phi map.
ANTENNA_GLOBAL_LOOKUP_MODE = os.environ.get(
    "TPC_DC_ANTENNA_LOOKUP_MODE", "dc_preprocessing"
)
if ANTENNA_GLOBAL_LOOKUP_MODE not in ("dc_preprocessing", "idealpadmap"):
    raise ValueError(f"Unknown ANTENNA_GLOBAL_LOOKUP_MODE={ANTENNA_GLOBAL_LOOKUP_MODE}")
ANTENNA_COMPARE_LOOKUP_MODES = True

# Optional empirical relative-response attachment. Raw antenna values are always
# preserved. Modes:
#   none       : no response scaling
#   outer_ring : one common factor chosen so A3 matches the empirical R1 extrapolation
#   one_scale  : one common robust factor from all four antenna rings
ANTENNA_ATTACH_MODE = os.environ.get("TPC_DC_ANTENNA_ATTACH_MODE", "none")
if ANTENNA_ATTACH_MODE not in ("none", "outer_ring", "one_scale"):
    raise ValueError(f"Unknown ANTENNA_ATTACH_MODE={ANTENNA_ATTACH_MODE}")
ANTENNA_ATTACH_FACTOR_MIN = 0.01
ANTENNA_ATTACH_FACTOR_MAX = 100.0

# ------------------------------------------------------------
# Output and batch
# ------------------------------------------------------------
def infer_run(path):
    m=re.search(r"run0*(\d{5})", str(path))
    return int(m.group(1)) if m else -1

def infer_window(path):
    m=re.search(r"win0*(\d+)", str(path))
    return int(m.group(1)) if m else -1

RUN_NUMBER = infer_run(INPUT_ROOT)
WINDOW_NUMBER = infer_window(INPUT_ROOT)

if RUN_NUMBER > 0 and WINDOW_NUMBER >= 0:
    default_tag = f"run{RUN_NUMBER}_win{WINDOW_NUMBER:02d}"
elif RUN_NUMBER > 0:
    default_tag = f"run{RUN_NUMBER}"
else:
    default_tag = Path(INPUT_ROOT).stem

RUN_TAG = os.environ.get("TPC_DC_TAG", default_tag)
IS_BATCH_CHILD = os.environ.get("TPC_DC_BATCH_CHILD", "0") == "1"

OUTPUT_DIR = Path(os.environ.get("TPC_DC_OUTPUT_DIR", "output_dc"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QA_PLOT_DIR = OUTPUT_DIR / f"qa_v24_{RUN_TAG}"
QA_PLOT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_ROOT = OUTPUT_DIR / f"TPC_DC_primary_density_v24_{RUN_TAG}.root"
OUTPUT_SUMMARY = OUTPUT_DIR / f"TPC_DC_summary_v24_{RUN_TAG}.csv"
OUTPUT_LAYER_PROFILE = OUTPUT_DIR / f"TPC_DC_layer_profile_v24_{RUN_TAG}.csv"
OUTPUT_STAGE_LAYER_PROFILE = OUTPUT_DIR / f"TPC_DC_layer_stages_v24_{RUN_TAG}.csv"
OUTPUT_PHI_PROFILE = OUTPUT_DIR / f"TPC_DC_phi_profile_v24_{RUN_TAG}.csv"

# All 57 local Takao-window files are flat in this directory.
BATCH_INPUT_GLOB = os.environ.get(
    "TPC_DC_BATCH_GLOB",
    "/media/yoren/T7_Shield/sPHENIX/tracking/dc/dc_run*.root"
)
BATCH_MAX_WORKERS = int(os.environ.get("TPC_DC_BATCH_WORKERS", "6"))
NOTEBOOK_FILE = Path(os.environ.get(
    "TPC_DC_NOTEBOOK",
    "TPC_IBF_PrimaryDensity_Extraction_v24_DC.ipynb"
))
REFERENCE_RUN = int(os.environ.get("TPC_DC_REFERENCE_RUN", "81558"))

print("Input:", INPUT_ROOT)
print("Run/window:", RUN_NUMBER, WINDOW_NUMBER)
print("Mapping:", MAPPING_DIR)
print("Gain map:", GAIN_MAP_FILE)
print("Main variant:", MAIN_VARIANT)
print("Defect-detection variant:", DEFECT_VARIANT)
print("Output:", OUTPUT_ROOT)
print("Batch glob:", BATCH_INPUT_GLOB)



## 1. Mapping geometry

The standard 48 TPC rows and the four R1 antenna rings are kept as **separate radial systems**.

For a standard TPC row, the radial collection width is the local layer pitch:
- interior: half the distance between the neighboring row centers;
- first row: distance to the second row;
- last row: distance to the previous row.

This is the same first/last convention used by `IdealPadMap::get_layer_thickness`. It avoids the v21 problem where the first standard R1 layer was widened toward the last antenna ring.

For φ, the mapping CSV stores the local CDB pad φ. The global side/sector transformation below follows the current `IdealPadMap` convention.


In [ ]:

def load_mapping(region):
    p = MAPPING_DIR / f"AutoPad-{region}-RevA.sch.ChannelMapping.csv"
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {p}. Set TPC_DC_MAPPING_DIR to "
            "DC_ML_processing/1_dc_preprocessing."
        )
    df=pd.read_csv(p)
    required={"Radius","Pad","PadName","PadR","PadPhi","FEE","FEE_Chan"}
    missing=required-set(df.columns)
    if missing:
        raise KeyError(f"{p} missing columns {sorted(missing)}")
    return df

mapping={m:load_mapping(f"R{m+1}") for m in range(3)}

def wrap_phi(phi):
    return (np.asarray(phi)+math.pi)%(2*math.pi)-math.pi

def centered_widths(x):
    """Pitch-cell width around centers without coupling separate detector systems."""
    x=np.asarray(x,float)
    if len(x)<2:
        return np.ones_like(x)
    w=np.empty_like(x)
    w[1:-1]=0.5*(x[2:]-x[:-2])
    w[0]=x[1]-x[0]
    w[-1]=x[-1]-x[-2]
    return np.abs(w)

def circular_pitch_widths(phi):
    """Per-pad local angular pitch from ordered local pad centers."""
    p=np.unwrap(np.asarray(phi,float))
    return centered_widths(p)

def row_geometry(df, radius_index):
    r=df[df["Radius"]==radius_index].copy().sort_values("Pad")
    if r.empty:
        raise KeyError(f"No Radius={radius_index}")
    return {
        "radius_index":int(radius_index),
        "r_cm":float(np.mean(r["PadR"]))/10.0,
        "n_pads":len(r),
        "phi_local":r["PadPhi"].to_numpy(float),
        "local_pad":r["Pad"].to_numpy(int),
        "pad_name":r["PadName"].astype(str).to_numpy(),
        "fee":r["FEE"].to_numpy(int),
        "fee_chan":r["FEE_Chan"].to_numpy(int),
    }

zigzag_rows={m:[row_geometry(mapping[m],il) for il in range(16)] for m in range(3)}
antenna_rows=[row_geometry(mapping[0],il) for il in (-4,-3,-2,-1)]

# Standard radial widths are derived separately within R1/R2/R3.
for m in range(3):
    centers=np.array([r["r_cm"] for r in zigzag_rows[m]],float)
    widths=centered_widths(centers)
    for il,row in enumerate(zigzag_rows[m]):
        row["dr_cm"]=float(widths[il])
        row["rlo"]=row["r_cm"]-0.5*row["dr_cm"]
        row["rhi"]=row["r_cm"]+0.5*row["dr_cm"]
        row["dphi_pad"]=circular_pitch_widths(row["phi_local"])

# Antenna rings are a separate radial system. Do not couple outer antenna width to R1.
ant_centers=np.array([r["r_cm"] for r in antenna_rows],float)
ant_widths=centered_widths(ant_centers)
for ia,row in enumerate(antenna_rows):
    row["dr_cm"]=float(ant_widths[ia])
    row["rlo"]=row["r_cm"]-0.5*row["dr_cm"]
    row["rhi"]=row["r_cm"]+0.5*row["dr_cm"]
    row["dphi_pad"]=circular_pitch_widths(row["phi_local"])

def global_phi_for_pad(row, side, sector, pad_index):
    """
    Reproduce current IdealPadMap side/sector transformation using
    local CDB phi from the mapping file.
    """
    npads=row["n_pads"]
    lookup=pad_index if side==0 else npads-1-pad_index
    cdb_phi=float(row["phi_local"][lookup])
    mapped_sector=(5 + N_SECTORS - (sector % N_SECTORS)) % N_SECTORS
    phi=((1.0 if side==1 else -1.0)*(cdb_phi-math.pi/2.0)
         + mapped_sector*math.pi/6.0)
    return float(wrap_phi(phi))

def global_phi_for_dc_preprocessing(row, side, sector, pad_index):
    """
    Phi convention used when the DC preprocessing filled the already-global
    phi-r histograms: mapped PadPhi plus the eBDC sector shift.

    This is a LOOKUP coordinate only. The final canonical r-phi output still
    uses global_phi_for_pad(), which follows the IdealPadMap convention.
    """
    ebdc = int(sector) + (12 if int(side)==1 else 0)
    local_phi = float(row["phi_local"][pad_index])
    return float(wrap_phi(local_phi + (ebdc - 12)*math.pi/6.0))

def antenna_lookup_phi(row, side, sector, pad_index, mode=ANTENNA_GLOBAL_LOOKUP_MODE):
    if mode == "dc_preprocessing":
        return global_phi_for_dc_preprocessing(row,side,sector,pad_index)
    if mode == "idealpadmap":
        return global_phi_for_pad(row,side,sector,pad_index)
    raise ValueError(mode)

def pad_area_cm2(row, pad_index):
    return row["r_cm"] * float(row["dphi_pad"][pad_index]) * row["dr_cm"]

print("Standard radial geometry:")
for m in range(3):
    rr=np.array([r["r_cm"] for r in zigzag_rows[m]])
    dr=np.array([r["dr_cm"] for r in zigzag_rows[m]])
    dp=np.concatenate([r["dphi_pad"] for r in zigzag_rows[m]])
    print(f" R{m+1}: r={rr[0]:.4f}..{rr[-1]:.4f} cm, "
          f"dr median={np.median(dr):.5f} cm, "
          f"dphi median={np.median(dp):.7f}")

print("Antenna radial geometry:")
for ia,row in enumerate(antenna_rows):
    print(f" A{ia}: r={row['r_cm']:.4f} cm, dr={row['dr_cm']:.4f} cm, "
          f"pads/sector={row['n_pads']}")

print("\nCritical v21 comparison:")
print(" first standard R1 dr in v24 =",zigzag_rows[0][0]["dr_cm"],"cm")
print(" nominal R1 pitch            =",zigzag_rows[0][1]["r_cm"]-zigzag_rows[0][0]["r_cm"],"cm")


## 2. Read synchronized DC ROOT input

Normal TPC rows use native `pad × layer` histograms. Antenna rings are reconstructed from the global `φ × r × frame` histograms because the present preprocessing does not yet write native antenna histograms.

Every DC quantity is normalized by its matching `nsamples` exposure.


In [ ]:

fin=root.TFile.Open(str(INPUT_ROOT),"READ")
if not fin or fin.IsZombie():
    raise OSError(f"Cannot open {INPUT_ROOT}")

def th2_to_numpy(h):
    nx,ny=h.GetNbinsX(),h.GetNbinsY()
    z=np.empty((ny,nx),float)
    for iy in range(ny):
        for ix in range(nx):
            z[iy,ix]=h.GetBinContent(ix+1,iy+1)
    return z

def project_xy_numpy(h3):
    nr,nphi=h3.GetNbinsY(),h3.GetNbinsX()
    out=np.zeros((nr,nphi),float)
    for ir in range(1,nr+1):
        for ip in range(1,nphi+1):
            out[ir-1,ip-1]=sum(h3.GetBinContent(ip,ir,iz)
                               for iz in range(1,h3.GetNbinsZ()+1))
    redges=np.array(
        [h3.GetYaxis().GetBinLowEdge(i) for i in range(1,nr+1)]
        +[h3.GetYaxis().GetBinUpEdge(nr)],float)
    pedges=np.array(
        [h3.GetXaxis().GetBinLowEdge(i) for i in range(1,nphi+1)]
        +[h3.GetXaxis().GetBinUpEdge(nphi)],float)
    return out,redges,pedges

def safe_ratio(num,den):
    return np.divide(num,den,out=np.full_like(num,np.nan,dtype=float),
                     where=np.isfinite(den)&(den>0))

def global_name(variant,side):
    return f"h_dc_phi_r_frame_{variant}_side{side}"

def native_name(variant,side,sector,module):
    return f"h_dc_pad_vs_layer_{variant}_side{side}_sec{sector:02d}_R{module+1}"

global_dc={}
for side in range(2):
    h_exp=fin.Get(f"h_dc_nsamples_phi_r_frame_side{side}")
    h_cnt=fin.Get(f"h_dc_sampleblocks_phi_r_frame_side{side}")
    if not h_exp or not h_cnt:
        raise KeyError(f"Missing global DC exposure/count side {side}")
    exp,r_edges_mm,phi_edges_02pi=project_xy_numpy(h_exp)
    cnt,_,_=project_xy_numpy(h_cnt)
    item={
        "nsamples":exp,
        "sampleblocks":cnt,
        "r_edges_mm":r_edges_mm,
        "phi_edges_02pi":phi_edges_02pi,
    }
    for variant in DC_VARIANTS:
        h=fin.Get(global_name(variant,side))
        if not h:
            raise KeyError(global_name(variant,side))
        q,_,_=project_xy_numpy(h)
        item[variant]=q
        item[f"{variant}_rate"]=safe_ratio(q,exp)
    global_dc[side]=item

native_dc={}
for side in range(2):
    for sector in range(12):
        for module in range(3):
            he=fin.Get(f"h_dc_nsamples_pad_vs_layer_side{side}_sec{sector:02d}_R{module+1}")
            hc=fin.Get(f"h_dc_sampleblocks_pad_vs_layer_side{side}_sec{sector:02d}_R{module+1}")
            if not he or not hc:
                raise KeyError(f"Missing native exposure/count s{side} sec{sector} R{module+1}")
            exp=th2_to_numpy(he)
            cnt=th2_to_numpy(hc)
            item={"nsamples":exp,"sampleblocks":cnt}
            for variant in DC_VARIANTS:
                h=fin.Get(native_name(variant,side,sector,module))
                if not h:
                    raise KeyError(native_name(variant,side,sector,module))
                q=th2_to_numpy(h)
                item[variant]=q
                item[f"{variant}_rate"]=safe_ratio(q,exp)
            native_dc[(side,module,sector)]=item

def norm_value(side,label):
    h=fin.Get("h_dc_norm")
    if not h:
        return np.nan
    iy=next((i for i in range(1,h.GetNbinsY()+1)
             if h.GetYaxis().GetBinLabel(i)==label),None)
    return h.GetBinContent(side+1,iy) if iy else np.nan

LIVE_FRAME_SLOTS=np.array([norm_value(s,"LIVE_FRAME_SLOTS") for s in range(2)])
LIVE_BCO_TICKS=np.array([norm_value(s,"LIVE_BCO_TICKS") for s in range(2)])

pGem=fin.Get("p_dc_gem_r1_current")
RUN_GEM_CURRENT=np.array([
    pGem.GetBinContent(1) if pGem else np.nan,
    pGem.GetBinContent(2) if pGem else np.nan
],float)

print("Live frame slots:",LIVE_FRAME_SLOTS)
print("R1 GEM current/load [South,North]:",RUN_GEM_CURRENT)

# Exposure masks are shared across all variants.
native_low_exposure={}
for key,item in native_dc.items():
    exp=item["nsamples"]
    good=exp[np.isfinite(exp)&(exp>0)]
    ref=float(np.median(good)) if len(good) else np.nan
    bad=(~np.isfinite(exp))|(exp<=0)
    if np.isfinite(ref):
        bad |= exp < MIN_EXPOSURE_FRACTION*ref
    native_low_exposure[key]=bad

fin.Close()



## 3. v20 dead-FEE / stripe / block detection and recovery

Defects are classified in native sector coordinates before any global \(r,\phi\) remapping. Low-exposure cells are excluded from the reference population.

The same structural masks are applied to every DC variant so signed/positive/strict comparisons are not biased by using different detector masks.


In [ ]:

def robust_scale(x):
    x=np.asarray(x,float)
    x=x[np.isfinite(x)]
    if not len(x):
        return 0.0
    med=np.median(x)
    mad=np.median(np.abs(x-med))
    return 1.4826*mad if mad>0 else np.std(x)

def clipped_values(x):
    x=np.asarray(x,float)
    x=x[np.isfinite(x)]
    if not len(x):
        return x
    med=np.median(x); sig=robust_scale(x)
    if CLIP_OUTLIERS and np.isfinite(sig) and sig>0:
        x=x[np.abs(x-med)<=CLIP_NSIGMA*sig]
    return x

def estimate_values(x,mode=None):
    mode=ADC_ESTIMATOR if mode is None else mode
    x=clipped_values(x)
    if not len(x):
        return np.nan
    if mode in ("raw","mean"):
        return float(np.mean(x))
    if mode=="sum":
        return float(np.sum(x))
    if mode=="median":
        return float(np.median(x))
    if mode=="trimmed_mean":
        xs=np.sort(x)
        ntrim=int(TRIM_FRACTION*len(xs))
        if 2*ntrim>=len(xs):
            return float(np.median(xs))
        return float(np.mean(xs[ntrim:len(xs)-ntrim]))
    raise ValueError(mode)

def contiguous_true_regions(mask):
    mask=np.asarray(mask,bool)
    out=[]; start=None
    for i,v in enumerate(mask):
        if v and start is None:
            start=i
        elif not v and start is not None:
            out.append((start,i)); start=None
    if start is not None:
        out.append((start,len(mask)))
    return out

def robust_dead_cell_candidates(adc):
    adc=np.asarray(adc,float)
    pos=np.where(np.isfinite(adc)&(adc>0),adc,np.nan)
    row=np.nanmedian(pos,axis=1)
    col=np.nanmedian(pos,axis=0)
    glob=np.nanmedian(pos)
    row=np.where(np.isfinite(row),row,glob)
    col=np.where(np.isfinite(col),col,glob)
    expected=np.maximum(row[:,None],col[None,:])
    candidate=(~np.isfinite(adc)) | (
        np.isfinite(expected)&(expected>0)&
        (adc<DEAD_CELL_REL_THRESHOLD*expected)
    )
    return candidate,expected

def radial_dead_fraction(adc,il):
    rows=[j for d in range(1,STRIPE_NEIGHBOR_LAYERS+1)
          for j in (il-d,il+d) if 0<=j<adc.shape[0]]
    if not rows:
        return np.zeros(adc.shape[1],bool)
    ref=np.nanmedian(
        np.where(np.asarray(adc[rows])>0,adc[rows],np.nan),axis=0
    )
    return (~np.isfinite(adc[il])) | (
        np.isfinite(ref)&(ref>0)&(adc[il]<STRIPE_REL_THRESHOLD*ref)
    )

def detect_dead_structures(adc,module,low_exposure=None):
    work=np.asarray(adc,float).copy()
    if low_exposure is not None:
        work[np.asarray(low_exposure,bool)]=np.nan

    candidate,expected=robust_dead_cell_candidates(work)
    nlayer,npad=work.shape
    mid=npad//2

    colfrac=np.mean(candidate,axis=0)
    fee_cols=colfrac>=DEAD_FEE_LAYER_FRACTION
    if FEE_BRIDGE_PADS>0:
        fee_cols=ndimage.binary_closing(
            fee_cols,structure=np.ones(2*FEE_BRIDGE_PADS+1)
        )
    fee_regions=[
        x for x in contiguous_true_regions(fee_cols)
        if x[1]-x[0]>=MIN_DEAD_FEE_WIDTH
    ]
    fee_region_mask=np.zeros_like(candidate)
    for lo,hi in fee_regions:
        fee_region_mask[:,lo:hi]=True
    fee_mask=fee_region_mask&candidate

    stripe_full=np.zeros_like(candidate)
    stripe_half=np.zeros_like(candidate)
    stripe_details=[]
    for il in range(nlayer):
        dead=radial_dead_fraction(work,il)
        usable=~fee_region_mask[il]
        if low_exposure is not None:
            usable &= ~np.asarray(low_exposure,bool)[il]
        if np.count_nonzero(usable)<4:
            continue

        frac=np.count_nonzero(dead&usable)/np.count_nonzero(usable)
        if frac>=FULL_STRIPE_FRACTION:
            stripe_full[il,usable]=True
            stripe_details.append(("full",il,float(frac)))
            continue

        if module==2 and ALLOW_R3_HALF_STRIPES:
            left=usable[:mid]; right=usable[mid:]
            lf=np.count_nonzero(dead[:mid]&left)/max(np.count_nonzero(left),1)
            rf=np.count_nonzero(dead[mid:]&right)/max(np.count_nonzero(right),1)
            if lf>=HALF_STRIPE_FRACTION and rf<=OTHER_HALF_MAX_DEAD_FRACTION:
                stripe_half[il,:mid]=left
                stripe_details.append(("half-left",il,float(lf)))
            elif rf>=HALF_STRIPE_FRACTION and lf<=OTHER_HALF_MAX_DEAD_FRACTION:
                stripe_half[il,mid:]=right
                stripe_details.append(("half-right",il,float(rf)))

    already=fee_mask|stripe_full|stripe_half
    remaining=candidate&~already
    labels,nlab=ndimage.label(remaining,structure=np.ones((3,3),int))
    block=np.zeros_like(candidate)
    block_details=[]
    for lab in range(1,nlab+1):
        yy,xx=np.where(labels==lab)
        if not len(xx):
            continue
        width=xx.max()-xx.min()+1
        height=yy.max()-yy.min()+1
        if (width>=MIN_DEAD_BLOCK_WIDTH and
            height>=MIN_DEAD_BLOCK_LAYERS and
            len(xx)>=MIN_DEAD_BLOCK_CELLS):
            block[yy,xx]=True
            block_details.append((
                int(yy.min()),int(yy.max()+1),
                int(xx.min()),int(xx.max()+1),int(len(xx))
            ))

    return {
        "candidate":candidate,
        "expected":expected,
        "fee":fee_mask,
        "fee_regions":fee_regions,
        "fee_region_mask":fee_region_mask,
        "stripe_full":stripe_full,
        "stripe_half":stripe_half,
        "stripe":stripe_full|stripe_half,
        "stripe_details":stripe_details,
        "block":block,
        "block_details":block_details,
        "col_dead_fraction":colfrac,
        "low_exposure":np.zeros_like(candidate) if low_exposure is None else np.asarray(low_exposure,bool),
    }

auto_masks={}
for key,item in native_dc.items():
    det=item[f"{DEFECT_VARIANT}_rate"].copy()
    auto_masks[key]=detect_dead_structures(det,key[1],native_low_exposure[key])

print("Dead-FEE regions:",sum(len(v["fee_regions"]) for v in auto_masks.values()))
print("Dead stripes:",sum(len(v["stripe_details"]) for v in auto_masks.values()))
print("Large dead blocks:",sum(len(v["block_details"]) for v in auto_masks.values()))
print("Low-exposure cells:",sum(np.count_nonzero(v["low_exposure"]) for v in auto_masks.values()))


In [ ]:

def local_robust_value(arr,il,ip,pad_radius,layer_radius,exclude=None):
    vals=[]
    for jl in range(max(0,il-layer_radius),min(arr.shape[0],il+layer_radius+1)):
        for jp in range(max(0,ip-pad_radius),min(arr.shape[1],ip+pad_radius+1)):
            if jl==il and jp==ip:
                continue
            if exclude is not None and exclude[jl,jp]:
                continue
            v=arr[jl,jp]
            if np.isfinite(v) and v>0:
                vals.append(v)
    return estimate_values(vals,"trimmed_mean") if len(vals)>=SINGLE_OUTLIER_MIN_NEIGHBORS else np.nan

def repair_fee_regions(adc,fee_mask,fee_regions,protected):
    out=adc.copy().astype(float)
    for lo,hi in fee_regions:
        for il,ip0 in zip(*np.where(fee_mask[:,lo:hi])):
            ip=ip0+lo
            left=[out[il,p] for p in range(max(0,lo-FEE_CONTEXT_PADS),lo)
                  if not protected[il,p] and np.isfinite(out[il,p]) and out[il,p]>0]
            right=[out[il,p] for p in range(hi,min(out.shape[1],hi+FEE_CONTEXT_PADS))
                   if not protected[il,p] and np.isfinite(out[il,p]) and out[il,p]>0]
            lv=estimate_values(left,"trimmed_mean")
            rv=estimate_values(right,"trimmed_mean")
            if np.isfinite(lv) and np.isfinite(rv):
                t=(ip-lo+1)/(hi-lo+1)
                out[il,ip]=(1-t)*lv+t*rv
            elif np.isfinite(lv):
                out[il,ip]=lv
            elif np.isfinite(rv):
                out[il,ip]=rv
    return out

def repair_mask_from_context(adc,mask,protected):
    out=adc.copy().astype(float)
    for il,ip in zip(*np.where(mask)):
        v=local_robust_value(out,il,ip,BLOCK_CONTEXT_PADS,BLOCK_CONTEXT_LAYERS,protected|mask)
        if np.isfinite(v):
            out[il,ip]=v
    return out

def recover_stripes(adc,stripe_mask,other_bad):
    out=adc.copy().astype(float)
    for il,ip in zip(*np.where(stripe_mask)):
        vals=[]
        for d in range(1,STRIPE_NEIGHBOR_LAYERS+1):
            for jl in (il-d,il+d):
                if 0<=jl<out.shape[0] and not other_bad[jl,ip]:
                    v=out[jl,ip]
                    if np.isfinite(v) and v>0:
                        vals.append(v)
        if vals:
            out[il,ip]=estimate_values(vals,"trimmed_mean")
        else:
            v=local_robust_value(
                out,il,ip,2,STRIPE_NEIGHBOR_LAYERS,
                other_bad|stripe_mask
            )
            if np.isfinite(v):
                out[il,ip]=v
    return out

def repair_single_outliers(adc,protected):
    out=adc.copy().astype(float)
    changed_mask=np.zeros_like(out,dtype=bool)
    for _ in range(SINGLE_OUTLIER_PASSES):
        new=out.copy()
        pass_changed=np.zeros_like(out,dtype=bool)
        for il in range(out.shape[0]):
            for ip in range(out.shape[1]):
                if protected[il,ip]:
                    continue
                ref=local_robust_value(
                    out,il,ip,SINGLE_OUTLIER_PAD_RADIUS,
                    SINGLE_OUTLIER_LAYER_RADIUS,protected
                )
                if not np.isfinite(ref) or ref<=0:
                    continue
                v=out[il,ip]
                bad=(not np.isfinite(v)) or v<=0 or \
                    v<SINGLE_DEAD_FACTOR*ref or v>SINGLE_HOT_FACTOR*ref
                if bad:
                    new[il,ip]=ref
                    pass_changed[il,ip]=True
        out=new
        changed_mask |= pass_changed
        if not np.any(pass_changed):
            break
    return out,changed_mask

def robust_smooth_native(adc):
    if not SMOOTH_NATIVE_MAPS:
        return adc.copy()
    out=adc.copy().astype(float)
    sm=out.copy()
    for il in range(out.shape[0]):
        for ip in range(out.shape[1]):
            vals=[]
            for jl in range(max(0,il-SMOOTH_LAYER_RADIUS),
                            min(out.shape[0],il+SMOOTH_LAYER_RADIUS+1)):
                for jp in range(max(0,ip-SMOOTH_PAD_RADIUS),
                                min(out.shape[1],ip+SMOOTH_PAD_RADIUS+1)):
                    v=out[jl,jp]
                    if np.isfinite(v) and v>0:
                        vals.append(v)
            ref=estimate_values(vals,"trimmed_mean")
            if np.isfinite(ref):
                sm[il,ip]=(1-SMOOTH_BLEND)*out[il,ip]+SMOOTH_BLEND*ref
    return sm

def recover_native_variant(variant):
    """
    Preserve every important recovery stage.
      measured   : input rate exactly as read
      exposure   : low-exposure cells set to NaN
      structure  : dead FEE + large block recovery
      stripes    : radial stripe recovery added
      outliers   : isolated hot/dead recovery added
      smoothed   : modest local native smoothing

    The legacy native_products tuple is still constructed below so the
    rest of the notebook remains backward-compatible.
    """
    stages={name:{} for name in (
        "measured","exposure","structure","stripes","outliers","smoothed"
    )}
    repaired_mask={}
    unstable_mask={}
    single_mask={}

    for key,item in native_dc.items():
        raw=item[f"{variant}_rate"].astype(float).copy()
        m=auto_masks[key]
        low=m["low_exposure"]
        stripe=m["stripe"]

        stages["measured"][key]=raw.copy()

        a0=raw.copy()
        a0[low]=np.nan
        stages["exposure"][key]=a0.copy()

        # Structural readout failures.
        a1=repair_fee_regions(
            a0,m["fee"],m["fee_regions"],
            stripe|m["block"]|low
        )
        a1=repair_mask_from_context(
            a1,m["block"],
            stripe|m["fee"]|low
        )
        stages["structure"][key]=a1.copy()

        # Radial dead stripes.
        other_bad=m["fee"]|m["block"]|low
        a2=recover_stripes(a1,stripe,other_bad)
        stages["stripes"][key]=a2.copy()

        # Isolated outliers.
        if REPAIR_SINGLE_OUTLIERS:
            a3,single=repair_single_outliers(a2,low)
        else:
            a3=a2.copy()
            single=np.zeros_like(a3,dtype=bool)
        stages["outliers"][key]=a3.copy()
        single_mask[key]=single

        # Mild detector-local smoothing. The final strong smoothing is not done
        # here; it is handled by the explicit two-source physics model.
        a4=robust_smooth_native(a3)
        stages["smoothed"][key]=a4.copy()

        repaired_mask[key]=m["fee"]|m["block"]|m["stripe"]|single
        unstable_mask[key]=m["fee"]|m["block"]|m["stripe"]|single|low

    return stages,repaired_mask,unstable_mask,single_mask

native_stages={}
native_products={}
native_single_masks={}

for variant in DC_VARIANTS:
    stages,rep,unst,single=recover_native_variant(variant)
    native_stages[variant]=stages
    native_single_masks[variant]=single
    # Legacy tuple used by the standard global map code.
    native_products[variant]=(
        stages["measured"],
        stages["smoothed"],
        rep,
        unst,
    )

print("Recovered native maps for:",", ".join(DC_VARIANTS))
print("Saved stages: measured -> exposure -> structure -> stripes -> outliers -> smoothed")



## 4. Gain map and standard-pad global density maps

For every native standard pad:

\[
\rho_{\rm DC} = \frac{Q/N_{\rm samples}}{r\,\Delta r\,\Delta\phi},
\qquad
\rho_{\rm primary} = \frac{\rho_{\rm DC}}{G_{\rm sector,layer}}.
\]

Measured and recovered maps are both retained. Coverage, repair fraction, and unstable fraction are propagated to the global map.


In [ ]:

gain_hist={}
if APPLY_STANDARD_PAD_GAIN:
    fg=root.TFile.Open(str(GAIN_MAP_FILE),"READ")
    if not fg or fg.IsZombie():
        raise OSError(f"Cannot open gain map {GAIN_MAP_FILE}")
    for side,name in [(0,GAIN_MAP_HIST_SIDE0),(1,GAIN_MAP_HIST_SIDE1)]:
        h=fg.Get(name)
        if not h:
            raise KeyError(name)
        h=h.Clone(name+"_clone")
        h.SetDirectory(0)
        gain_hist[side]=h
    fg.Close()

def gain_for(side,sector,module,local_layer):
    if not APPLY_STANDARD_PAD_GAIN:
        return 1.0
    glayer=7+16*module+local_layer
    h=gain_hist[side]
    bx=h.GetXaxis().FindBin(sector+0.5)
    by=h.GetYaxis().FindBin((glayer-GAIN_LAYER_OFFSET)+0.5)
    g=float(h.GetBinContent(bx,by))
    return g if np.isfinite(g) and g>0 else np.nan

# Optional static layer-response correction derived from the cross-run pass.
layer_response_lookup={}
if APPLY_LAYER_RESPONSE_CORRECTION:
    if LAYER_RESPONSE_FILE.exists():
        _lr=pd.read_csv(LAYER_RESPONSE_FILE)
        required={"side","global_layer","correction"}
        missing=required-set(_lr.columns)
        if missing:
            raise KeyError(f"{LAYER_RESPONSE_FILE} missing {sorted(missing)}")
        for _,row in _lr.iterrows():
            layer_response_lookup[
                (int(row["side"]),int(row["global_layer"]))
            ]=float(row["correction"])
        print("Loaded layer-response correction:",LAYER_RESPONSE_FILE)
    else:
        print("WARNING: layer-response correction requested but file missing:",
              LAYER_RESPONSE_FILE)

def layer_response_for(side,module,local_layer):
    if not APPLY_LAYER_RESPONSE_CORRECTION:
        return 1.0
    glayer=7+16*module+local_layer
    c=layer_response_lookup.get((side,glayer),1.0)
    return c if np.isfinite(c) and c>0 else 1.0

phi_edges=np.linspace(PHI_MIN,PHI_MAX,GLOBAL_PHI_BINS+1)
phi_centers=0.5*(phi_edges[:-1]+phi_edges[1:])
r_edges=np.arange(SOURCE_R_MIN_CM,SOURCE_R_MAX_CM+GLOBAL_DR_CM*1.001,GLOBAL_DR_CM)
if r_edges[-1]<SOURCE_R_MAX_CM:
    r_edges=np.r_[r_edges,SOURCE_R_MAX_CM]
r_edges[-1]=SOURCE_R_MAX_CM
r_centers=0.5*(r_edges[:-1]+r_edges[1:])
GRID_SHAPE=(len(r_centers),len(phi_centers))

def circular_delta(a,b):
    return np.arctan2(np.sin(a-b),np.cos(a-b))

def cell_bins(rlo,rhi,phi_center,dphi):
    ir=np.where((r_centers>=rlo)&(r_centers<rhi))[0]
    ip=np.where(np.abs(circular_delta(phi_centers,phi_center))<=0.5*dphi)[0]
    return ir,ip

def accumulate_cell(sum_map,count_map,value,rlo,rhi,phi,dphi,weight=1.0):
    if not np.isfinite(value):
        return
    ir,ip=cell_bins(rlo,rhi,phi,dphi)
    if not len(ir) or not len(ip):
        return
    sum_map[np.ix_(ir,ip)] += value*weight
    count_map[np.ix_(ir,ip)] += weight

def build_standard_global(side,variant,native_maps,repaired_mask,unstable_mask):
    dc_sum=np.zeros(GRID_SHAPE)
    pr_sum=np.zeros(GRID_SHAPE)
    dc_count=np.zeros(GRID_SHAPE)
    pr_count=np.zeros(GRID_SHAPE)
    repaired_sum=np.zeros(GRID_SHAPE)
    unstable_sum=np.zeros(GRID_SHAPE)

    for module in range(3):
        response=layer_response_for(side,module,0)  # overwritten per layer below
        for sector in range(12):
            key=(side,module,sector)
            a=native_maps[key]
            rep=repaired_mask[key]
            unst=unstable_mask[key]

            for il,row in enumerate(zigzag_rows[module]):
                g=gain_for(side,sector,module,il)
                response=layer_response_for(side,module,il)
                for ip in range(row["n_pads"]):
                    v=a[il,ip]
                    if not np.isfinite(v):
                        continue
                    dphi=float(row["dphi_pad"][ip])
                    area=pad_area_cm2(row,ip)
                    if not np.isfinite(area) or area<=0:
                        continue

                    rho=v/area
                    prho=rho/g if np.isfinite(g) and g>0 else np.nan
                    if np.isfinite(prho) and np.isfinite(response) and response>0:
                        prho=prho/response

                    phi=global_phi_for_pad(row,side,sector,ip)
                    ir,ib=cell_bins(row["rlo"],row["rhi"],phi,dphi)
                    if not len(ir) or not len(ib):
                        continue

                    dc_sum[np.ix_(ir,ib)] += rho
                    dc_count[np.ix_(ir,ib)] += 1.0

                    if np.isfinite(prho):
                        pr_sum[np.ix_(ir,ib)] += prho
                        pr_count[np.ix_(ir,ib)] += 1.0

                    repaired_sum[np.ix_(ir,ib)] += float(rep[il,ip])
                    unstable_sum[np.ix_(ir,ib)] += float(unst[il,ip])

    def average(s,c):
        out=np.full(GRID_SHAPE,np.nan)
        good=c>0
        out[good]=s[good]/c[good]
        return out

    return {
        "dc":average(dc_sum,dc_count),
        "primary":average(pr_sum,pr_count),
        "coverage":dc_count,
        "primary_coverage":pr_count,
        "repair_fraction":average(repaired_sum,dc_count),
        "unstable_fraction":average(unstable_sum,dc_count),
    }

standard_maps={}
for variant in DC_VARIANTS:
    measured,recovered,rep,unst=native_products[variant]
    standard_maps[variant]={"measured":{},"recovered":{}}
    for side in range(2):
        standard_maps[variant]["measured"][side]=build_standard_global(
            side,variant,measured,rep,unst
        )
        standard_maps[variant]["recovered"][side]=build_standard_global(
            side,variant,recovered,rep,unst
        )

# Intermediate global maps for MAIN_VARIANT. These make every recovery step
# inspectable rather than collapsing everything into "measured" vs "recovered".
_main_rep=native_products[MAIN_VARIANT][2]
_main_unst=native_products[MAIN_VARIANT][3]
main_stage_standard_maps={}
for stage,nmaps in native_stages[MAIN_VARIANT].items():
    main_stage_standard_maps[stage]={}
    for side in range(2):
        main_stage_standard_maps[stage][side]=build_standard_global(
            side,MAIN_VARIANT,nmaps,_main_rep,_main_unst
        )

print("Built standard-pad global maps for all variants.")
print("Built intermediate global stages for",MAIN_VARIANT,
      ":",", ".join(main_stage_standard_maps))



## 5. Antenna rings: area-integrated extraction and recovery

v21 sampled a single coarse global DC bin at each antenna-pad center. v22 instead **integrates all coarse global bins whose centers fall inside the antenna collection cell**, separately summing charge and exposure:

\[
\bar Q_{\rm antenna} =
\frac{\sum_{\rm cell}Q}{\sum_{\rm cell}N_{\rm samples}}.
\]

The four antenna rings are treated as their own radial-pitch system. Their response is **not** divided by the normal 48-layer gain map.

A ring/φ robust outlier mask and local recovery are applied, while both measured and recovered antenna products are saved. This is still provisional until native antenna histograms are added to the C++ preprocessing.


In [ ]:
def global_cell_integral(item,arrname,rlo,rhi,phi_center,dphi):
    """
    Integrate global coarse bins by bin-center membership inside one antenna cell.
    r is stored in mm, phi in [0,2pi).
    """
    redges=item["r_edges_mm"]/10.0
    rcen=0.5*(redges[:-1]+redges[1:])
    pedges=item["phi_edges_02pi"]
    pcen=0.5*(pedges[:-1]+pedges[1:])
    pcen_wrapped=wrap_phi(pcen)

    ir=np.where((rcen>=rlo)&(rcen<rhi))[0]
    ip=np.where(np.abs(circular_delta(pcen_wrapped,phi_center))<=0.5*dphi)[0]
    if not len(ir) or not len(ip):
        return np.nan,0
    vals=item[arrname][np.ix_(ir,ip)]
    return float(np.nansum(vals)), int(vals.size)

def build_antenna_records(side,lookup_mode=ANTENNA_GLOBAL_LOOKUP_MODE):
    """
    Extract antenna DC from the already-global DC histogram.

    phi_lookup: coordinate used to FIND the cell in the preprocessing histogram.
    phi:        canonical IdealPadMap-style coordinate used in the final map.

    Keeping these separate fixes the v23 convention mismatch.
    """
    gitem=global_dc[side]
    rows=[]
    for ia,row in enumerate(antenna_rows):
        for sector in range(12):
            for ip in range(row["n_pads"]):
                phi_output=global_phi_for_pad(row,side,sector,ip)
                phi_lookup=antenna_lookup_phi(row,side,sector,ip,lookup_mode)
                dphi=float(row["dphi_pad"][ip])
                area=pad_area_cm2(row,ip)

                exp,nbin=global_cell_integral(
                    gitem,"nsamples",row["rlo"],row["rhi"],phi_lookup,dphi
                )
                cnt,_=global_cell_integral(
                    gitem,"sampleblocks",row["rlo"],row["rhi"],phi_lookup,dphi
                )

                rec={
                    "ring":ia,
                    "sector":sector,
                    "pad":ip,
                    "phi":phi_output,
                    "phi_lookup":phi_lookup,
                    "lookup_mode":lookup_mode,
                    "r_cm":row["r_cm"],
                    "rlo":row["rlo"],
                    "rhi":row["rhi"],
                    "dphi":dphi,
                    "area_cm2":area,
                    "nsamples":exp,
                    "sampleblocks":cnt,
                    "global_bins":nbin,
                }
                for variant in DC_VARIANTS:
                    q,_=global_cell_integral(
                        gitem,variant,row["rlo"],row["rhi"],phi_lookup,dphi
                    )
                    rate=q/exp if np.isfinite(exp) and exp>0 else np.nan
                    rec[f"{variant}_rate"]=rate
                    rec[f"{variant}_density"]=rate/area if np.isfinite(rate) else np.nan
                rows.append(rec)
    return pd.DataFrame(rows)

# Main extraction uses the convention requested in configuration.
antenna_df={s:build_antenna_records(s,ANTENNA_GLOBAL_LOOKUP_MODE) for s in range(2)}

# Explicit lookup-convention QA. This is cheap (384 antenna pads/side) and is
# intentionally kept even after we choose a default convention.
antenna_lookup_qa_rows=[]
if ANTENNA_COMPARE_LOOKUP_MODES:
    for lookup_mode in ("dc_preprocessing","idealpadmap"):
        for side in range(2):
            qdf=build_antenna_records(side,lookup_mode)
            for ring in range(4):
                q=qdf[qdf["ring"]==ring]
                antenna_lookup_qa_rows.append({
                    "run":RUN_NUMBER,
                    "window":WINDOW_NUMBER,
                    "side":side,
                    "lookup_mode":lookup_mode,
                    "ring":ring,
                    "r_cm":float(np.nanmedian(q["r_cm"])),
                    "median_rate":float(np.nanmedian(q[f"{MAIN_VARIANT}_rate"])),
                    "median_density":float(np.nanmedian(q[f"{MAIN_VARIANT}_density"])),
                    "median_exposure":float(np.nanmedian(q["nsamples"])),
                })
antenna_lookup_qa_df=pd.DataFrame(antenna_lookup_qa_rows)

def antenna_mask_and_recover(df,variant):
    d=df.copy().sort_values(["ring","phi"]).reset_index(drop=True)
    density=d[f"{variant}_density"].to_numpy(float)
    exp=d["nsamples"].to_numpy(float)

    bad=(~np.isfinite(exp))|(exp<=0)|(~np.isfinite(density))

    # Exposure relative to each ring.
    for ring in range(4):
        idx=np.where(d["ring"].to_numpy()==ring)[0]
        goodexp=exp[idx][np.isfinite(exp[idx])&(exp[idx]>0)]
        if len(goodexp):
            eref=np.median(goodexp)
            bad[idx] |= exp[idx] < ANTENNA_MIN_EXPOSURE_FRACTION*eref

    # Signed pedestal pathology is useful even when another variant is studied.
    signed=d["ped60_signed_rate"].to_numpy(float)
    bad |= np.isfinite(signed)&(signed<0)

    # Loose robust ring-level outlier rejection.
    for ring in range(4):
        idx=np.where(d["ring"].to_numpy()==ring)[0]
        vals=density[idx]
        good=np.isfinite(vals)&(vals>0)&(~bad[idx])
        if np.count_nonzero(good)<8:
            continue
        lv=np.log(vals[good])
        med=np.median(lv)
        mad=1.4826*np.median(np.abs(lv-med))
        if mad<=0:
            continue
        local_bad=np.zeros(len(idx),bool)
        local_bad[good]=np.abs(np.log(vals[good])-med)>ANTENNA_LOG_MAD_NSIGMA*mad
        bad[idx[local_bad]]=True

    # Recover in a 4 x Nphi representation, preserving circular phi.
    ring_arrays=[]
    ring_bad=[]
    ring_indices=[]
    for ring in range(4):
        idx=np.where(d["ring"].to_numpy()==ring)[0]
        idx=idx[np.argsort(d.loc[idx,"phi"].to_numpy())]
        ring_indices.append(idx)
        ring_arrays.append(density[idx].copy())
        ring_bad.append(bad[idx].copy())

    nphi=min(len(x) for x in ring_arrays)
    arr=np.vstack([x[:nphi] for x in ring_arrays])
    msk=np.vstack([x[:nphi] for x in ring_bad])
    out=arr.copy()

    for _ in range(ANTENNA_REPAIR_PASSES):
        new=out.copy()
        changed=0
        for ir in range(4):
            for ip in range(nphi):
                if not msk[ir,ip]:
                    continue
                vals=[]
                for dr in range(-ANTENNA_REPAIR_RING_RADIUS,ANTENNA_REPAIR_RING_RADIUS+1):
                    jr=ir+dr
                    if jr<0 or jr>=4:
                        continue
                    for dp in range(-ANTENNA_REPAIR_PHI_RADIUS,ANTENNA_REPAIR_PHI_RADIUS+1):
                        jp=(ip+dp)%nphi
                        if jr==ir and jp==ip:
                            continue
                        if msk[jr,jp]:
                            continue
                        v=out[jr,jp]
                        if np.isfinite(v) and v>0:
                            vals.append(v)
                ref=estimate_values(vals,"trimmed_mean")
                if np.isfinite(ref):
                    new[ir,ip]=ref
                    changed+=1
        out=new
        if changed==0:
            break

    recovered=density.copy()
    for ring in range(4):
        idx=ring_indices[ring][:nphi]
        recovered[idx]=out[ring]

    return bad,recovered

antenna_products={}
for variant in DC_VARIANTS:
    antenna_products[variant]={}
    for side in range(2):
        bad,recovered=antenna_mask_and_recover(antenna_df[side],variant)
        antenna_products[variant][side]={
            "bad":bad,
            "measured":antenna_df[side][f"{variant}_density"].to_numpy(float),
            "recovered":recovered,
        }

for side in range(2):
    print(f"\nAntenna side {side} (lookup={ANTENNA_GLOBAL_LOOKUP_MODE})")
    d=antenna_df[side]
    for ring in range(4):
        idx=np.where(d["ring"].to_numpy()==ring)[0]
        vals=antenna_products[MAIN_VARIANT][side]["recovered"][idx]
        nbad=np.count_nonzero(antenna_products[MAIN_VARIANT][side]["bad"][idx])
        print(f" ring {ring}: r={np.median(d.loc[idx,'r_cm']):.3f} cm, "
              f"median recovered={np.nanmedian(vals):.6g}, masked={nbad}/{len(idx)}, "
              f"global bins/pad median={np.median(d.loc[idx,'global_bins']):.1f}")

if len(antenna_lookup_qa_df):
    print("\nAntenna lookup-convention QA (before recovery):")
    print(antenna_lookup_qa_df.to_string(index=False))



## 6. Combine standard and antenna maps

The standard-pad map remains the calibrated primary-density proxy. Antenna bins are kept as an uncalibrated low-r extension and explicitly tagged with an antenna mask.

For physics fits, the notebook uses standard R1 by default. Antenna points are compared against the standard-R1 extrapolation rather than forcing them into the fit.


In [ ]:

def antenna_to_grid(side,variant,mode):
    d=antenna_df[side]
    vals=antenna_products[variant][side][mode]
    sm=np.zeros(GRID_SHAPE)
    cnt=np.zeros(GRID_SHAPE)
    mask=np.zeros(GRID_SHAPE,bool)

    for i,rec in d.iterrows():
        v=vals[i]
        if not np.isfinite(v):
            continue
        ir,ip=cell_bins(
            float(rec.rlo),float(rec.rhi),
            float(rec.phi),float(rec.dphi)
        )
        if not len(ir) or not len(ip):
            continue
        sm[np.ix_(ir,ip)] += v
        cnt[np.ix_(ir,ip)] += 1
        mask[np.ix_(ir,ip)] = True

    out=np.full(GRID_SHAPE,np.nan)
    good=cnt>0
    out[good]=sm[good]/cnt[good]
    return out,cnt,mask

combined_maps={}
for variant in DC_VARIANTS:
    combined_maps[variant]={"measured":{},"recovered":{}}
    for mode in ("measured","recovered"):
        for side in range(2):
            std=standard_maps[variant][mode][side]
            dc=std["dc"].copy()
            primary=std["primary"].copy()
            coverage=std["coverage"].copy()

            ant,acov,amask=antenna_to_grid(side,variant,mode)
            take=np.isfinite(ant)&(~np.isfinite(dc))
            dc[take]=ant[take]
            # antenna has no normal gain correction
            primary[take]=ant[take]
            coverage[take]=acov[take]

            combined_maps[variant][mode][side]={
                "dc":dc,
                "primary":primary,
                "coverage":coverage,
                "antenna_mask":amask,
                "repair_fraction":std["repair_fraction"],
                "unstable_fraction":std["unstable_fraction"],
            }

print("Combined standard + antenna maps built.")


## 7. Physical radial profiles and empirical \(1/r^\alpha\)

The fine \(r\times\phi\) grid remains useful for visualization and PHGarfield-oriented output, but it is **not** used to count radial measurements.

For radial QA we go back to detector coordinates and form one robust value for each of the 48 physical TPC rows. The R1 power-law fit uses the first `R1_FIT_N_LAYERS` physical rows.

This removes the v22 pseudo-replication where a single physical row was painted into several 0.1-cm grid bins and then treated as several independent fit points.


In [ ]:

def robust_profile_r(values,coverage=None,rmin=None,rmax=None):
    """
    Fine-grid display profile. This is useful for plotting, but alpha fits below
    use the 48 PHYSICAL TPC layer centers, not repeated 0.1-cm grid bins.
    """
    out=np.full(values.shape[0],np.nan)
    err=np.full(values.shape[0],np.nan)
    n=np.zeros(values.shape[0],int)
    for ir,row in enumerate(values):
        if rmin is not None and r_centers[ir]<rmin:
            continue
        if rmax is not None and r_centers[ir]>rmax:
            continue
        good=np.isfinite(row)
        if coverage is not None:
            good &= coverage[ir]>0
        vals=row[good]
        vals=vals[np.isfinite(vals)]
        if not len(vals):
            continue
        med=np.median(vals)
        mad=1.4826*np.median(np.abs(vals-med))
        out[ir]=med
        err[ir]=max(mad/max(np.sqrt(len(vals)),1.0),0.01*max(abs(med),1e-12))
        n[ir]=len(vals)
    return out,err,n

def robust_location(values,clip_nsigma=5.0):
    v=np.asarray(values,float)
    v=v[np.isfinite(v)]
    if not len(v):
        return np.nan,np.nan,0
    for _ in range(2):
        med=np.median(v)
        mad=1.4826*np.median(np.abs(v-med))
        if not np.isfinite(mad) or mad<=0:
            break
        keep=np.abs(v-med)<=clip_nsigma*mad
        if np.all(keep) or np.count_nonzero(keep)<5:
            break
        v=v[keep]
    med=np.median(v)
    mad=1.4826*np.median(np.abs(v-med))
    err=max(mad/max(np.sqrt(len(v)),1.0),0.01*max(abs(med),1e-12))
    return float(med),float(err),int(len(v))

def build_physical_layer_profile(side,variant,stage):
    """
    One robust value per actual TPC row. This is the authoritative radial QA.
    It avoids painting one physical row into several 0.1-cm grid bins.
    """
    maps=native_stages[variant][stage]
    rows=[]
    for module in range(3):
        for il,row in enumerate(zigzag_rows[module]):
            dcvals=[]
            prvals=[]
            gains=[]
            response=layer_response_for(side,module,il)
            for sector in range(12):
                a=maps[(side,module,sector)]
                g=gain_for(side,sector,module,il)
                if not np.isfinite(g) or g<=0:
                    continue
                vv=np.asarray(a[il,:row["n_pads"]],float)
                areas=np.array([pad_area_cm2(row,ip) for ip in range(row["n_pads"])],float)
                good=np.isfinite(vv)&np.isfinite(areas)&(areas>0)
                if not np.any(good):
                    continue
                dc=vv[good]/areas[good]
                pr=dc/g
                if np.isfinite(response) and response>0:
                    pr=pr/response
                dcvals.extend(dc.tolist())
                prvals.extend(pr.tolist())
                gains.append(g)

            dcmed,dcerr,ndc=robust_location(dcvals)
            prmed,prerr,npr=robust_location(prvals)
            rows.append({
                "run":RUN_NUMBER,
                "window":WINDOW_NUMBER,
                "side":side,
                "variant":variant,
                "stage":stage,
                "module":module,
                "local_layer":il,
                "global_layer":7+16*module+il,
                "r_cm":row["r_cm"],
                "dc_density":dcmed,
                "dc_err":dcerr,
                "primary_density":prmed,
                "primary_err":prerr,
                "n_dc":ndc,
                "n_primary":npr,
                "median_gain":float(np.nanmedian(gains)) if len(gains) else np.nan,
                "layer_response_applied":response,
            })
    return pd.DataFrame(rows)

# Fine-grid profiles kept for display / ROOT compatibility.
profile_cache={}
for variant in DC_VARIANTS:
    for mode in ("measured","recovered"):
        for quantity in ("dc","primary"):
            for side in range(2):
                item=combined_maps[variant][mode][side]
                p,e,n=robust_profile_r(item[quantity],item["coverage"])
                profile_cache[(variant,mode,quantity,side)]=(p,e,n)

# Physical row profiles for every variant in the two principal modes.
layer_profile_frames=[]
for variant in DC_VARIANTS:
    for mode,stage in (("measured","measured"),("recovered","smoothed")):
        for side in range(2):
            d=build_physical_layer_profile(side,variant,stage)
            d["mode"]=mode
            layer_profile_frames.append(d)

layer_profile_df=pd.concat(layer_profile_frames,ignore_index=True)

# Preserve ALL intermediate recovery stages for the preferred pedestal treatment.
stage_layer_frames=[]
for stage in ("measured","exposure","structure","stripes","outliers","smoothed"):
    for side in range(2):
        d=build_physical_layer_profile(side,MAIN_VARIANT,stage)
        stage_layer_frames.append(d)
stage_layer_profile_df=pd.concat(stage_layer_frames,ignore_index=True)

def fit_powerlaw_layers(df,quantity="primary",n_layers=R1_FIT_N_LAYERS):
    ycol=f"{quantity}_density"
    ecol=f"{quantity}_err"
    d=df[
        (df["module"]==0)&
        np.isfinite(df[ycol])&
        (df[ycol]>0)&
        (df["r_cm"]>=R1_FIT_RMIN_CM)&
        (df["r_cm"]<=R1_FIT_RMAX_CM)
    ].sort_values("local_layer").copy()

    if n_layers>0:
        d=d.iloc[:n_layers]

    rr=d["r_cm"].to_numpy(float)
    yy=d[ycol].to_numpy(float)
    ee=d[ecol].to_numpy(float)

    if len(rr)<3:
        return {
            "alpha":np.nan,"alpha_err":np.nan,"rho_ref":np.nan,
            "r_ref":np.nan,"n":len(rr),"r":rr,"y":yy,
            "yerr":ee,"fit":np.full_like(rr,np.nan)
        }

    rref=float(rr[0])
    x=np.log(rref/rr)
    ly=np.log(yy)
    scale=np.clip(np.divide(ee,yy,out=np.ones_like(ee)*0.05,where=yy>0),0.02,0.30)

    p0=np.array([np.log(np.median(yy)),ALPHA_INITIAL],float)
    lo=np.array([-np.inf,ALPHA_BOUNDS[0]])
    hi=np.array([ np.inf,ALPHA_BOUNDS[1]])

    def resid(p):
        return (ly-(p[0]+p[1]*x))/scale

    res=least_squares(resid,p0,bounds=(lo,hi),loss="soft_l1",f_scale=1.0)
    alpha=float(res.x[1])
    rho_ref=float(np.exp(res.x[0]))

    alpha_err=np.nan
    try:
        j=res.jac
        dof=max(len(rr)-len(res.x),1)
        cov=np.linalg.pinv(j.T@j)*(2*res.cost/dof)
        alpha_err=float(np.sqrt(max(cov[1,1],0)))
    except Exception:
        pass

    fit=rho_ref*(rref/rr)**alpha
    return {
        "alpha":alpha,"alpha_err":alpha_err,"rho_ref":rho_ref,
        "r_ref":rref,"n":len(rr),"r":rr,"y":yy,
        "yerr":ee,"fit":fit
    }

# R1 alpha is now fitted to 12 physical rows, not 12 fine-grid bins.
alpha_results=[]
layer_fit_cache={}
for variant in DC_VARIANTS:
    for mode in ("measured","recovered"):
        stage="measured" if mode=="measured" else "smoothed"
        for quantity in ("dc","primary"):
            for side in range(2):
                d=layer_profile_df[
                    (layer_profile_df["variant"]==variant)&
                    (layer_profile_df["mode"]==mode)&
                    (layer_profile_df["side"]==side)
                ]
                fit=fit_powerlaw_layers(d,quantity)
                layer_fit_cache[(variant,mode,quantity,side)]=fit
                alpha_results.append({
                    "run":RUN_NUMBER,
                    "window":WINDOW_NUMBER,
                    "variant":variant,
                    "mode":mode,
                    "quantity":quantity,
                    "side":side,
                    "alpha":fit["alpha"],
                    "alpha_err":fit["alpha_err"],
                    "rho_ref":fit["rho_ref"],
                    "r_ref":fit["r_ref"],
                    "nfit":fit["n"],
                })

alpha_df=pd.DataFrame(alpha_results)

print("\nR1 power-law comparison (12 PHYSICAL R1 layers)")
show=alpha_df[
    (alpha_df["quantity"]=="primary")&
    (alpha_df["mode"]=="recovered")
]
print(show[["variant","side","alpha","alpha_err","nfit"]].to_string(index=False))


## 8. Smooth two-source radial/phi model

The recovered detector map can still contain stable module and layer response structure. We therefore keep the recovered data untouched and fit a separate smooth physics hypothesis:

\[
\rho(r,\phi)=
A_{\rm coll}(R_0/r)^{\alpha_{\rm coll}}
+
A_{\rm beam}(R_0/r)^{\alpha_{\rm beam}}
[1+\epsilon\cos(\phi-\phi_0)].
\]

Defaults are `alpha_collision = 1.23` and `alpha_beam = 2.0`. The first term is azimuthally flat. The second contains only one preferred direction.

The fit also contains multiplicative R2/R3 response factors relative to R1. These are detector nuisance parameters, not part of the final smooth physical density. This is especially useful for diagnosing the current R3 discontinuity.


In [ ]:

# ============================================================
# Smooth two-source model and detector-response diagnostics
# ============================================================

def build_layer_phi_profile(side,variant=MAIN_VARIANT,stage="smoothed",quantity="primary",
                            nphi=PHYSICS_MODEL_PHI_BINS):
    """
    Build one robust datum per physical TPC row and coarse phi bin.

    This avoids oversampling the 0.1-cm display grid and is the input to the
    smooth two-source model. Antenna pads are excluded here because their
    absolute gain response is not yet calibrated.
    """
    maps=native_stages[variant][stage]
    pedges=np.linspace(PHI_MIN,PHI_MAX,nphi+1)
    pcent=0.5*(pedges[:-1]+pedges[1:])
    rows=[]

    for module in range(3):
        for il,row in enumerate(zigzag_rows[module]):
            buckets=[[] for _ in range(nphi)]
            response=layer_response_for(side,module,il)

            for sector in range(12):
                a=maps[(side,module,sector)]
                g=gain_for(side,sector,module,il)
                if not np.isfinite(g) or g<=0:
                    continue

                for ip in range(row["n_pads"]):
                    v=float(a[il,ip])
                    if not np.isfinite(v):
                        continue

                    area=pad_area_cm2(row,ip)
                    if not np.isfinite(area) or area<=0:
                        continue

                    rho=v/area
                    if quantity=="primary":
                        rho=rho/g
                        if np.isfinite(response) and response>0:
                            rho=rho/response

                    phi=global_phi_for_pad(row,side,sector,ip)
                    ib=np.searchsorted(pedges,phi,side="right")-1
                    if ib==nphi:
                        ib=nphi-1
                    if 0<=ib<nphi and np.isfinite(rho):
                        buckets[ib].append(rho)

            for ib,vals in enumerate(buckets):
                if len(vals)<PHYSICS_MODEL_MIN_PER_BIN:
                    continue
                med,err,n=robust_location(vals)
                if not np.isfinite(med):
                    continue
                rows.append({
                    "run":RUN_NUMBER,
                    "window":WINDOW_NUMBER,
                    "side":side,
                    "variant":variant,
                    "stage":stage,
                    "quantity":quantity,
                    "module":module,
                    "local_layer":il,
                    "global_layer":7+16*module+il,
                    "r_cm":row["r_cm"],
                    "phi":pcent[ib],
                    "density":med,
                    "density_err":err,
                    "n":n,
                })

    return pd.DataFrame(rows)

def _physics_components(r,alpha_collision,alpha_beam):
    r=np.asarray(r,float)
    fc=(PHYSICS_MODEL_RREF_CM/r)**float(alpha_collision)
    fb=(PHYSICS_MODEL_RREF_CM/r)**float(alpha_beam)
    return fc,fb

def evaluate_two_source_model(r,phi,fit,include_module_scale=False,module=None):
    r=np.asarray(r,float)
    phi=np.asarray(phi,float)
    fc,fb=_physics_components(r,fit["alpha_collision"],fit["alpha_beam"])

    beam_mod=1.0+fit["beam_epsilon"]*np.cos(phi-fit["beam_phi0"])
    val=fit["A_collision"]*fc + fit["A_beam"]*fb*beam_mod

    if include_module_scale:
        if module is None:
            raise ValueError("module is required when include_module_scale=True")
        m=np.asarray(module,int)
        scales=np.ones_like(val,dtype=float)
        scales[m==1]=fit["module_scale_R2"]
        scales[m==2]=fit["module_scale_R3"]
        val=val*scales

    return val

def fit_two_source_model(df):
    """
    Joint robust fit of:
      collision: flat phi, alpha free in COLLISION_ALPHA_BOUNDS
      beam background: n=1 phi harmonic, alpha free in BEAM_ALPHA_BOUNDS
      R2/R3 multiplicative detector-response scales relative to R1

    The alpha values start at 1.23 and 2.0 by default but are NOT fixed.
    R2/R3 scale factors remain explicit nuisance parameters.
    """
    d=df.copy()
    good=(
        np.isfinite(d["density"])&
        (d["density"]>0)&
        np.isfinite(d["density_err"])&
        (d["r_cm"]>=PHYSICS_MODEL_RMIN_CM)&
        (d["r_cm"]<=PHYSICS_MODEL_RMAX_CM)
    )
    d=d[good].copy()

    empty={
        "success":False,
        "A_collision":np.nan,
        "A_beam":np.nan,
        "alpha_collision":np.nan,
        "alpha_collision_err":np.nan,
        "alpha_beam":np.nan,
        "alpha_beam_err":np.nan,
        "beam_epsilon":np.nan,
        "beam_phi0":np.nan,
        "module_scale_R1":1.0,
        "module_scale_R2":np.nan,
        "module_scale_R3":np.nan,
        "cost_ndf":np.nan,
        "n":len(d),
    }
    if len(d)<50:
        return empty

    r=d["r_cm"].to_numpy(float)
    phi=d["phi"].to_numpy(float)
    y=d["density"].to_numpy(float)
    err=d["density_err"].to_numpy(float)
    module=d["module"].to_numpy(int)

    scaled=y*(r/PHYSICS_MODEL_RREF_CM)**COLLISION_ALPHA_INITIAL
    amp=max(float(np.nanmedian(scaled)),1e-12)

    # log(Ac), log(Ab), alpha_c, alpha_b, qeps, phi0, log(sR2), log(sR3)
    p0=np.array([
        math.log(0.7*amp),
        math.log(0.3*amp),
        COLLISION_ALPHA_INITIAL,
        BEAM_ALPHA_INITIAL,
        np.arctanh(0.10/PHYSICS_MODEL_EPS_MAX),
        0.0,
        0.0,
        0.0,
    ],float)

    lscale_min=math.log(PHYSICS_MODEL_SCALE_MIN)
    lscale_max=math.log(PHYSICS_MODEL_SCALE_MAX)

    lo=np.array([
        -40,-40,
        COLLISION_ALPHA_BOUNDS[0],BEAM_ALPHA_BOUNDS[0],
        -4.0,-math.pi,
        lscale_min,lscale_min
    ],float)
    hi=np.array([
         40, 40,
        COLLISION_ALPHA_BOUNDS[1],BEAM_ALPHA_BOUNDS[1],
         4.0, math.pi,
        lscale_max,lscale_max
    ],float)

    siglog=np.clip(
        np.divide(err,y,out=np.ones_like(err)*0.10,where=y>0),
        PHYSICS_MODEL_ERROR_FLOOR,
        0.50
    )

    def unpack(p):
        Ac=math.exp(p[0])
        Ab=math.exp(p[1])
        ac=float(p[2])
        ab=float(p[3])
        eps=PHYSICS_MODEL_EPS_MAX*math.tanh(p[4])
        phi0=float(p[5])
        s2=math.exp(p[6])
        s3=math.exp(p[7])
        return Ac,Ab,ac,ab,eps,phi0,s2,s3

    def prediction(p):
        Ac,Ab,ac,ab,eps,phi0,s2,s3=unpack(p)
        fc,fb=_physics_components(r,ac,ab)
        phys=Ac*fc + Ab*fb*(1.0+eps*np.cos(phi-phi0))
        scales=np.ones_like(phys)
        scales[module==1]=s2
        scales[module==2]=s3
        return phys*scales

    def resid(p):
        pred=np.clip(prediction(p),1e-30,np.inf)
        return (np.log(y)-np.log(pred))/siglog

    res=least_squares(
        resid,p0,bounds=(lo,hi),
        loss="soft_l1",f_scale=1.0,
        max_nfev=8000
    )

    Ac,Ab,ac,ab,eps,phi0,s2,s3=unpack(res.x)
    if eps < 0:
        eps=-eps
        phi0=float(wrap_phi(phi0+math.pi))

    npar=len(res.x)
    ndf=max(len(d)-npar,1)

    # Approximate local parameter uncertainties for QA. With robust loss these
    # are diagnostics, not precision statistical confidence intervals.
    perr=np.full(len(res.x),np.nan)
    try:
        jtj=res.jac.T@res.jac
        cov=np.linalg.pinv(jtj)*(2*res.cost/ndf)
        diag=np.diag(cov)
        perr=np.sqrt(np.where(diag>=0,diag,np.nan))
    except Exception:
        pass

    return {
        "success":bool(res.success),
        "A_collision":Ac,
        "A_beam":Ab,
        "alpha_collision":ac,
        "alpha_collision_err":float(perr[2]) if len(perr)>2 else np.nan,
        "alpha_beam":ab,
        "alpha_beam_err":float(perr[3]) if len(perr)>3 else np.nan,
        "beam_epsilon":eps,
        "beam_phi0":phi0,
        "module_scale_R1":1.0,
        "module_scale_R2":s2,
        "module_scale_R3":s3,
        "cost_ndf":float(2*res.cost/ndf),
        "n":int(len(d)),
        "message":res.message,
        "collision_alpha_at_bound":bool(
            abs(ac-COLLISION_ALPHA_BOUNDS[0])<1e-3 or
            abs(ac-COLLISION_ALPHA_BOUNDS[1])<1e-3
        ),
        "beam_alpha_at_bound":bool(
            abs(ab-BEAM_ALPHA_BOUNDS[0])<1e-3 or
            abs(ab-BEAM_ALPHA_BOUNDS[1])<1e-3
        ),
    }

def module_scale_for(fit,module):
    if module==0:
        return 1.0
    if module==1:
        return float(fit["module_scale_R2"])
    if module==2:
        return float(fit["module_scale_R3"])
    return np.nan

# Build the model input and fit separately for each side.
layer_phi_df={}
physics_models={}
for side in range(2):
    d=build_layer_phi_profile(side,MAIN_VARIANT,"smoothed","primary")
    layer_phi_df[side]=d
    physics_models[side]=fit_two_source_model(d)

print("\nTwo-source smooth model")
for side in range(2):
    f=physics_models[side]
    print(
        f" side {side}: "
        f"A_collision={f['A_collision']:.6g}, "
        f"A_beam={f['A_beam']:.6g}, "
        f"alpha_coll={f['alpha_collision']:.4f} +/- {f['alpha_collision_err']:.4f}, "
        f"alpha_beam={f['alpha_beam']:.4f} +/- {f['alpha_beam_err']:.4f}, "
        f"eps={f['beam_epsilon']:.4f}, "
        f"phi0={f['beam_phi0']:.4f}, "
        f"R2 scale={f['module_scale_R2']:.4f}, "
        f"R3 scale={f['module_scale_R3']:.4f}, "
        f"cost/ndf={f['cost_ndf']:.3f}, n={f['n']}, "
        f"alpha_bounds_hit=(coll:{f.get('collision_alpha_at_bound',False)}, "
        f"beam:{f.get('beam_alpha_at_bound',False)})"
    )

# ------------------------------------------------------------
# Build smooth / equalized / residual 2D products
# ------------------------------------------------------------
physics_products={}
standard_r_bounds={}
for module in range(3):
    standard_r_bounds[module]=(
        min(row["rlo"] for row in zigzag_rows[module]),
        max(row["rhi"] for row in zigzag_rows[module])
    )

def module_from_radius(r):
    for module,(lo,hi) in standard_r_bounds.items():
        if lo<=r<hi:
            return module
    return -1

for side in range(2):
    fit=physics_models[side]
    recovered=combined_maps[MAIN_VARIANT]["recovered"][side]["primary"]

    equalized=np.full_like(recovered,np.nan,float)
    model_standard=np.full_like(recovered,np.nan,float)
    model_extrapolated=np.full_like(recovered,np.nan,float)

    for ir,r in enumerate(r_centers):
        # Smooth physical extrapolation is kept separately all the way inward.
        model_extrapolated[ir,:]=evaluate_two_source_model(
            np.full_like(phi_centers,r,dtype=float),
            phi_centers,
            fit,
            include_module_scale=False
        )

        module=module_from_radius(r)
        if module<0:
            continue

        scale=module_scale_for(fit,module)
        row=recovered[ir]
        good=np.isfinite(row)
        if np.isfinite(scale) and scale>0:
            equalized[ir,good]=row[good]/scale

        model_standard[ir,:]=evaluate_two_source_model(
            np.full_like(phi_centers,r,dtype=float),
            phi_centers,
            fit,
            include_module_scale=False
        )

    ratio=np.full_like(equalized,np.nan)
    good=np.isfinite(equalized)&np.isfinite(model_standard)&(model_standard>0)
    ratio[good]=equalized[good]/model_standard[good]

    residual=np.full_like(equalized,np.nan)
    residual[good]=equalized[good]-model_standard[good]

    physics_products[side]={
        "equalized_recovered":equalized,
        "smooth_model_standard":model_standard,
        "smooth_model_extrapolated":model_extrapolated,
        "ratio_to_model":ratio,
        "residual_to_model":residual,
    }

# ------------------------------------------------------------
# Physical-layer radial diagnostics
# ------------------------------------------------------------
physics_layer_frames=[]
for side in range(2):
    fit=physics_models[side]
    d=layer_profile_df[
        (layer_profile_df["variant"]==MAIN_VARIANT)&
        (layer_profile_df["mode"]=="recovered")&
        (layer_profile_df["side"]==side)
    ].copy()

    d["module_scale"]=[
        module_scale_for(fit,int(m)) for m in d["module"]
    ]
    d["primary_equalized"]=d["primary_density"]/d["module_scale"]

    fc,fb=_physics_components(d["r_cm"].to_numpy(float),fit["alpha_collision"],fit["alpha_beam"])
    d["model_radial_isotropic"]=(
        fit["A_collision"]*fc + fit["A_beam"]*fb
    )
    d["ratio_before_equalization"]=(
        d["primary_density"]/d["model_radial_isotropic"]
    )
    d["ratio_after_equalization"]=(
        d["primary_equalized"]/d["model_radial_isotropic"]
    )
    physics_layer_frames.append(d)

physics_layer_df=pd.concat(physics_layer_frames,ignore_index=True)

# ------------------------------------------------------------
# Phi QA after removing the smooth radial dependence and module scale.
# This makes the n=1 structure visible without mixing it with r.
# ------------------------------------------------------------
phi_profile_frames=[]
for side in range(2):
    fit=physics_models[side]
    d=layer_phi_df[side].copy()

    scales=np.array([module_scale_for(fit,int(m)) for m in d["module"]],float)
    equalized=d["density"].to_numpy(float)/scales
    fc,fb=_physics_components(d["r_cm"].to_numpy(float),fit["alpha_collision"],fit["alpha_beam"])
    iso=fit["A_collision"]*fc + fit["A_beam"]*fb
    raw_ratio=equalized/iso

    full_model=evaluate_two_source_model(
        d["r_cm"].to_numpy(float),
        d["phi"].to_numpy(float),
        fit,
        include_module_scale=False
    )
    pred_ratio=full_model/iso

    d["equalized_density"]=equalized
    d["phi_ratio_data"]=raw_ratio
    d["phi_ratio_model"]=pred_ratio

    for phi in sorted(d["phi"].unique()):
        s=d[np.isclose(d["phi"],phi)]
        med,err,n=robust_location(s["phi_ratio_data"].to_numpy(float))
        pmed,_,_=robust_location(s["phi_ratio_model"].to_numpy(float))
        phi_profile_frames.append(pd.DataFrame([{
            "run":RUN_NUMBER,
            "window":WINDOW_NUMBER,
            "side":side,
            "phi":float(phi),
            "ratio_data":med,
            "ratio_err":err,
            "ratio_model":pmed,
            "n":n,
        }]))

phi_profile_df=pd.concat(phi_profile_frames,ignore_index=True)

print("\nModule-response diagnostic from smooth model")
for side in range(2):
    f=physics_models[side]
    print(
        f" side {side}: R1=1.0000, "
        f"R2={f['module_scale_R2']:.4f}, "
        f"R3={f['module_scale_R3']:.4f}"
    )

## 9. Standard-R1 extrapolation and antenna response

The antenna pads are still not placed on the normal 48-layer gain scale. They therefore remain a separate diagnostic.

Two curves are shown against the antenna rings:
- the empirical standard-R1 power-law extrapolation;
- the smooth two-source physical expectation.

Neither uses the antenna values as fit constraints.


In [ ]:
def standard_r1_prediction(side,variant=MAIN_VARIANT,mode="recovered",quantity="primary"):
    fit=layer_fit_cache[(variant,mode,quantity,side)]
    d=layer_profile_df[
        (layer_profile_df["variant"]==variant)&
        (layer_profile_df["mode"]==mode)&
        (layer_profile_df["side"]==side)
    ].copy()
    return fit,d

def antenna_values_to_grid(side,values):
    d=antenna_df[side]
    sm=np.zeros(GRID_SHAPE)
    cnt=np.zeros(GRID_SHAPE)
    mask=np.zeros(GRID_SHAPE,bool)
    for i,rec in d.iterrows():
        v=float(values[i])
        if not np.isfinite(v):
            continue
        ir,ip=cell_bins(float(rec.rlo),float(rec.rhi),float(rec.phi),float(rec.dphi))
        if not len(ir) or not len(ip):
            continue
        sm[np.ix_(ir,ip)] += v
        cnt[np.ix_(ir,ip)] += 1.0
        mask[np.ix_(ir,ip)] = True
    out=np.full(GRID_SHAPE,np.nan)
    good=cnt>0
    out[good]=sm[good]/cnt[good]
    return out,cnt,mask

antenna_ratio_rows=[]
antenna_ring_summary={}
antenna_attach_factor={}
antenna_attached_values={}
antenna_attached_grid={}
primary_continuous_map={}

for side in range(2):
    fit,d=standard_r1_prediction(side)
    adf=antenna_df[side]
    vals=antenna_products[MAIN_VARIANT][side]["recovered"]
    antenna_ring_summary[side]=[]

    ring_expected=[]
    ring_measured=[]
    for ring in range(4):
        idx=np.where(adf["ring"].to_numpy()==ring)[0]
        rring=float(np.nanmedian(adf.loc[idx,"r_cm"]))
        med=float(np.nanmedian(vals[idx]))
        expected=(
            float(fit["rho_ref"]*(fit["r_ref"]/rring)**fit["alpha"])
            if np.isfinite(fit["alpha"]) else np.nan
        )
        ratio=med/expected if np.isfinite(expected) and expected>0 else np.nan
        ring_expected.append(expected)
        ring_measured.append(med)
        antenna_ring_summary[side].append((rring,med,expected,ratio))

    # One response factor only. We never scale rings independently.
    if ANTENNA_ATTACH_MODE=="none":
        factor=1.0
    elif ANTENNA_ATTACH_MODE=="outer_ring":
        factor=(ring_expected[3]/ring_measured[3]
                if np.isfinite(ring_expected[3]) and np.isfinite(ring_measured[3]) and ring_measured[3]>0
                else np.nan)
    elif ANTENNA_ATTACH_MODE=="one_scale":
        ratios=np.array(ring_expected,float)/np.array(ring_measured,float)
        good=np.isfinite(ratios)&(ratios>0)
        factor=float(np.nanmedian(ratios[good])) if np.any(good) else np.nan
    else:
        raise ValueError(ANTENNA_ATTACH_MODE)

    if not np.isfinite(factor):
        factor=1.0
    factor=float(np.clip(factor,ANTENNA_ATTACH_FACTOR_MIN,ANTENNA_ATTACH_FACTOR_MAX))
    antenna_attach_factor[side]=factor
    attached=vals*factor
    antenna_attached_values[side]=attached
    agrid,acov,amask=antenna_values_to_grid(side,attached)
    antenna_attached_grid[side]=agrid

    # Data-like continuous map: standard recovered primary + optionally attached antenna.
    base=combined_maps[MAIN_VARIANT]["recovered"][side]["primary"].copy()
    if ANTENNA_ATTACH_MODE!="none":
        take=amask&np.isfinite(agrid)
        base[take]=agrid[take]
    primary_continuous_map[side]=base

    for ring in range(4):
        idx=np.where(adf["ring"].to_numpy()==ring)[0]
        rring=float(np.nanmedian(adf.loc[idx,"r_cm"]))
        med=ring_measured[ring]
        expected=ring_expected[ring]
        attached_med=float(np.nanmedian(attached[idx]))
        ratio=med/expected if np.isfinite(expected) and expected>0 else np.nan
        attached_ratio=(attached_med/expected
                        if np.isfinite(expected) and expected>0 else np.nan)
        antenna_ratio_rows.append({
            "run":RUN_NUMBER,
            "window":WINDOW_NUMBER,
            "side":side,
            "ring":ring,
            "r_cm":rring,
            "antenna_rate":float(np.nanmedian(adf.loc[idx,f"{MAIN_VARIANT}_rate"])),
            "area_cm2":float(np.nanmedian(adf.loc[idx,"area_cm2"])),
            "antenna_density":med,
            "antenna_density_attached":attached_med,
            "r1_extrapolated":expected,
            "antenna_over_r1_extrap":ratio,
            "attached_over_r1_extrap":attached_ratio,
            "attach_factor":factor,
            "attach_mode":ANTENNA_ATTACH_MODE,
            "lookup_mode":ANTENNA_GLOBAL_LOOKUP_MODE,
        })

antenna_ratio_df=pd.DataFrame(antenna_ratio_rows)
print("\nAntenna / empirical standard-R1 extrapolation")
print(antenna_ratio_df.to_string(index=False))
print("\nAntenna common response attachment:")
for side in range(2):
    print(f" side {side}: mode={ANTENNA_ATTACH_MODE}, factor={antenna_attach_factor[side]:.6g}")

# Geometry continuity QA: these collection cells are already nearly touching.
outer_ant_rhi=max(row["rhi"] for row in antenna_rows)
first_r1_rlo=min(row["rlo"] for row in zigzag_rows[0])
print(f"Antenna-to-R1 radial cell-edge gap = {first_r1_rlo-outer_ant_rhi:.5f} cm "
      f"({10*(first_r1_rlo-outer_ant_rhi):.3f} mm)")


## 10. QA plots

The single-file QA now emphasizes the problems visible in the current DC maps:

- every native recovery stage;
- physical-row radial profiles before/after each recovery step;
- DC versus gain-corrected primary radial shape;
- recovered primary before/after R2/R3 response equalization;
- smooth two-source radial model and row/model ratios;
- normalized \(\phi\) profile and its \(n=1\) description;
- equalized 2D data, smooth 2D model, and data/model ratio;
- all pedestal definitions in R1;
- antenna response against both extrapolations.


In [ ]:

import matplotlib.pyplot as plt

QA_EXAMPLE=(0,0,0)  # side,module,sector

def finite_limits(a,qlo=2,qhi=98):
    v=np.asarray(a,float)
    v=v[np.isfinite(v)]
    if not len(v):
        return None,None
    return tuple(np.percentile(v,[qlo,qhi]))

def plot_native_stages(side,module,sector,variant=MAIN_VARIANT):
    key=(side,module,sector)
    stages=("measured","structure","stripes","outliers","smoothed")
    arrays=[native_stages[variant][s][key] for s in stages]
    m=auto_masks[key]
    mask=(m["fee"]|m["stripe"]|m["block"]|m["low_exposure"]).astype(float)

    lo,hi=finite_limits(arrays[-1])
    fig,ax=plt.subplots(2,3,figsize=(16,8),constrained_layout=True)

    for a,title,axis in zip(arrays,stages,ax.flat[:5]):
        im=axis.imshow(a,aspect="auto",origin="lower",vmin=lo,vmax=hi)
        axis.set_title(title)
        axis.set_xlabel("pad")
        axis.set_ylabel("local layer")
        fig.colorbar(im,ax=axis)

    im=ax.flat[5].imshow(mask,aspect="auto",origin="lower",vmin=0,vmax=1)
    ax.flat[5].set_title("FEE/stripe/block/exposure mask")
    ax.flat[5].set_xlabel("pad")
    ax.flat[5].set_ylabel("local layer")
    fig.colorbar(im,ax=ax.flat[5])

    fig.suptitle(
        f"Native recovery stages s{side} R{module+1} sector {sector:02d} {variant}"
    )
    path=QA_PLOT_DIR/f"native_stages_s{side}_R{module+1}_sec{sector:02d}_{variant}.png"
    fig.savefig(path,dpi=150)
    print("saved",path)
    print(" fee regions:",m["fee_regions"])
    print(" stripe details:",m["stripe_details"])
    print(" block details:",m["block_details"])
    return fig

def plot_radial_diagnostics(side,variant=MAIN_VARIANT):
    fig,ax=plt.subplots(2,2,figsize=(14,10),constrained_layout=True)

    # 1) Every recovery stage, physical primary rows.
    for stage in ("measured","structure","stripes","outliers","smoothed"):
        d=stage_layer_profile_df[
            (stage_layer_profile_df["side"]==side)&
            (stage_layer_profile_df["stage"]==stage)
        ]
        ax[0,0].plot(d["r_cm"],d["primary_density"],".-",label=stage)
    ax[0,0].set_title("Physical-row primary profile: recovery stages")
    ax[0,0].set_xlabel("r [cm]")
    ax[0,0].set_ylabel("primary-density proxy")
    ax[0,0].legend(fontsize=8)
    ax[0,0].grid(True,alpha=0.25)

    # 2) DC vs primary, normalized to their own R1 medians.
    d=layer_profile_df[
        (layer_profile_df["variant"]==variant)&
        (layer_profile_df["mode"]=="recovered")&
        (layer_profile_df["side"]==side)
    ].copy()
    r1=d["module"]==0
    dc0=np.nanmedian(d.loc[r1,"dc_density"])
    pr0=np.nanmedian(d.loc[r1,"primary_density"])
    if np.isfinite(dc0) and dc0!=0:
        ax[0,1].plot(d["r_cm"],d["dc_density"]/dc0,".-",label="DC / R1 median")
    if np.isfinite(pr0) and pr0!=0:
        ax[0,1].plot(d["r_cm"],d["primary_density"]/pr0,".-",label="primary / R1 median")
    ax[0,1].axhline(1.0,ls="--",lw=1)
    ax[0,1].set_title("Gain diagnostic: DC vs primary shape")
    ax[0,1].set_xlabel("r [cm]")
    ax[0,1].set_ylabel("normalized density")
    ax[0,1].legend()
    ax[0,1].grid(True,alpha=0.25)

    # 3) Recovered primary before/after module equalization plus smooth physics model.
    pdx=physics_layer_df[physics_layer_df["side"]==side].copy()
    fit=physics_models[side]
    rr=np.linspace(PHYSICS_MODEL_RMIN_CM,PHYSICS_MODEL_RMAX_CM,400)
    fc,fb=_physics_components(rr,fit["alpha_collision"],fit["alpha_beam"])
    radial_model=fit["A_collision"]*fc+fit["A_beam"]*fb

    ax[1,0].errorbar(
        pdx["r_cm"],pdx["primary_density"],yerr=pdx["primary_err"],
        fmt=".",label="recovered physical rows"
    )
    ax[1,0].plot(
        pdx["r_cm"],pdx["primary_equalized"],".",
        label="after R2/R3 response equalization"
    )
    ax[1,0].plot(
        rr,radial_model,"-",
        label=(
            f"smooth 2-source radial model "
            f"(a_coll={fit['alpha_collision']:.2f}, a_beam={fit['alpha_beam']:.2f})"
        )
    )
    ax[1,0].set_title(
        f"Smooth radial model; R2 scale={fit['module_scale_R2']:.3f}, "
        f"R3 scale={fit['module_scale_R3']:.3f}"
    )
    ax[1,0].set_xlabel("r [cm]")
    ax[1,0].set_ylabel("primary-density proxy")
    ax[1,0].legend(fontsize=8)
    ax[1,0].grid(True,alpha=0.25)

    # 4) Response ratio before and after module equalization.
    ax[1,1].plot(
        pdx["r_cm"],pdx["ratio_before_equalization"],".-",
        label="before module equalization"
    )
    ax[1,1].plot(
        pdx["r_cm"],pdx["ratio_after_equalization"],".-",
        label="after module equalization"
    )
    ax[1,1].axhline(1.0,ls="--",lw=1)
    ax[1,1].set_title("Physical row / smooth radial model")
    ax[1,1].set_xlabel("r [cm]")
    ax[1,1].set_ylabel("ratio")
    ax[1,1].legend()
    ax[1,1].grid(True,alpha=0.25)

    fig.suptitle(f"Radial diagnostics side {side}, {variant}")
    path=QA_PLOT_DIR/f"radial_diagnostics_side{side}_{variant}.png"
    fig.savefig(path,dpi=160)
    print("saved",path)
    return fig

def plot_phi_diagnostics(side):
    d=phi_profile_df[phi_profile_df["side"]==side].sort_values("phi")
    fit=physics_models[side]

    fig,ax=plt.subplots(1,2,figsize=(14,5.5),constrained_layout=True)

    ax[0].errorbar(
        d["phi"],d["ratio_data"],yerr=d["ratio_err"],
        fmt=".",label="data after radial/module normalization"
    )
    ax[0].plot(d["phi"],d["ratio_model"],"-",label="two-source n=1 model")
    ax[0].axhline(1.0,ls="--",lw=1)
    ax[0].set_xlabel("phi")
    ax[0].set_ylabel("density / isotropic radial component")
    ax[0].set_title(
        f"Phi profile side {side}: eps={fit['beam_epsilon']:.3f}, "
        f"phi0={fit['beam_phi0']:.3f}"
    )
    ax[0].legend()
    ax[0].grid(True,alpha=0.25)

    # Residual from n=1 description.
    residual=d["ratio_data"]-d["ratio_model"]
    ax[1].plot(d["phi"],residual,".-")
    ax[1].axhline(0.0,ls="--",lw=1)
    ax[1].set_xlabel("phi")
    ax[1].set_ylabel("data - n=1 model")
    ax[1].set_title("Phi residual")
    ax[1].grid(True,alpha=0.25)

    path=QA_PLOT_DIR/f"phi_diagnostics_side{side}_{MAIN_VARIANT}.png"
    fig.savefig(path,dpi=160)
    print("saved",path)
    return fig

def plot_physics_2d(side):
    p=physics_products[side]
    fig,ax=plt.subplots(1,3,figsize=(18,5.5),constrained_layout=True)

    lo,hi=finite_limits(p["equalized_recovered"])
    im=ax[0].imshow(
        p["equalized_recovered"],origin="lower",aspect="auto",
        extent=[PHI_MIN,PHI_MAX,r_edges[0],r_edges[-1]],
        vmin=lo,vmax=hi
    )
    ax[0].set_title("Recovered data after module equalization")
    ax[0].set_xlabel("phi"); ax[0].set_ylabel("r [cm]")
    fig.colorbar(im,ax=ax[0])

    im=ax[1].imshow(
        p["smooth_model_standard"],origin="lower",aspect="auto",
        extent=[PHI_MIN,PHI_MAX,r_edges[0],r_edges[-1]]
    )
    ax[1].set_title("Smooth two-source model")
    ax[1].set_xlabel("phi"); ax[1].set_ylabel("r [cm]")
    fig.colorbar(im,ax=ax[1])

    im=ax[2].imshow(
        p["ratio_to_model"],origin="lower",aspect="auto",
        extent=[PHI_MIN,PHI_MAX,r_edges[0],r_edges[-1]],
        vmin=0.7,vmax=1.3
    )
    ax[2].set_title("Equalized data / smooth model")
    ax[2].set_xlabel("phi"); ax[2].set_ylabel("r [cm]")
    fig.colorbar(im,ax=ax[2])

    fig.suptitle(f"Smooth primary-density diagnostics side {side}")
    path=QA_PLOT_DIR/f"physics_model_2d_side{side}.png"
    fig.savefig(path,dpi=160)
    print("saved",path)
    return fig

def plot_variant_r1_comparison(side):
    fig,ax=plt.subplots(figsize=(9,6))
    for variant in DC_VARIANTS:
        d=layer_profile_df[
            (layer_profile_df["variant"]==variant)&
            (layer_profile_df["mode"]=="recovered")&
            (layer_profile_df["side"]==side)&
            (layer_profile_df["module"]==0)
        ].sort_values("local_layer")
        fit=layer_fit_cache[(variant,"recovered","primary",side)]
        ax.plot(d["r_cm"],d["primary_density"],".-",label=f"{variant}")
        if np.isfinite(fit["alpha"]):
            ax.plot(
                fit["r"],fit["fit"],"--",
                label=f"{variant} R1 alpha={fit['alpha']:.2f}"
            )
    ax.set_xlabel("r [cm]")
    ax.set_ylabel("primary-density proxy")
    ax.set_title(f"Pedestal/noise definition comparison, R1 side {side}")
    ax.grid(True,alpha=0.25)
    ax.legend(fontsize=8)
    path=QA_PLOT_DIR/f"variant_r1_comparison_side{side}.png"
    fig.savefig(path,dpi=160,bbox_inches="tight")
    print("saved",path)
    return fig

def plot_antenna_extrapolation(side):
    fit,_=standard_r1_prediction(side)
    fig,ax=plt.subplots(figsize=(8,5.5))

    d=layer_profile_df[
        (layer_profile_df["variant"]==MAIN_VARIANT)&
        (layer_profile_df["mode"]=="recovered")&
        (layer_profile_df["side"]==side)&
        (layer_profile_df["module"]==0)
    ].sort_values("local_layer")

    ax.errorbar(
        d["r_cm"],d["primary_density"],yerr=d["primary_err"],
        fmt=".",label="standard R1 physical rows"
    )

    for ring,(rval,med,expected,ratio) in enumerate(antenna_ring_summary[side]):
        ax.scatter([rval],[med],s=55,label=f"antenna A{ring} raw, ratio={ratio:.2f}")
        attached=antenna_ratio_df[
            (antenna_ratio_df["side"]==side)&(antenna_ratio_df["ring"]==ring)
        ]
        if len(attached) and ANTENNA_ATTACH_MODE!="none":
            av=float(attached.iloc[0]["antenna_density_attached"])
            ar=float(attached.iloc[0]["attached_over_r1_extrap"])
            ax.scatter([rval],[av],marker="x",s=65,
                       label=f"A{ring} attached, ratio={ar:.2f}")

    xr=np.linspace(22.5,41,250)
    if np.isfinite(fit["alpha"]):
        yr=fit["rho_ref"]*(fit["r_ref"]/xr)**fit["alpha"]
        ax.plot(xr,yr,"--",label=f"empirical R1 extrap alpha={fit['alpha']:.2f}")

    # Also show the smooth two-source physical extrapolation, but do not use
    # antenna points to constrain it.
    f=physics_models[side]
    fc,fb=_physics_components(xr,f["alpha_collision"],f["alpha_beam"])
    yphys=f["A_collision"]*fc+f["A_beam"]*fb
    ax.plot(xr,yphys,"-",label="two-source isotropic expectation")

    ax.set_xlabel("r [cm]")
    ax.set_ylabel("primary-density proxy")
    ax.set_title(f"Antenna vs standard-pad extrapolations side {side}\nlookup={ANTENNA_GLOBAL_LOOKUP_MODE}, attach={ANTENNA_ATTACH_MODE}")
    ax.grid(True,alpha=0.25)
    ax.legend(fontsize=8)
    path=QA_PLOT_DIR/f"antenna_vs_r1_side{side}.png"
    fig.savefig(path,dpi=150,bbox_inches="tight")
    print("saved",path)
    return fig

def plot_antenna_diagnostics(side):
    d=antenna_ratio_df[antenna_ratio_df["side"]==side].sort_values("ring")
    q=antenna_lookup_qa_df[antenna_lookup_qa_df["side"]==side] if len(antenna_lookup_qa_df) else pd.DataFrame()
    fig,ax=plt.subplots(2,2,figsize=(13,9),constrained_layout=True)

    ax[0,0].plot(d["r_cm"],d["antenna_rate"],"o-")
    ax[0,0].set_title("Antenna Q/Nsamples before area correction")
    ax[0,0].set_xlabel("r [cm]"); ax[0,0].set_ylabel("DC rate")
    ax[0,0].grid(True,alpha=0.25)

    ax[0,1].plot(d["r_cm"],d["area_cm2"],"o-")
    ax[0,1].set_title("Antenna pitch-cell area used")
    ax[0,1].set_xlabel("r [cm]"); ax[0,1].set_ylabel("area [cm2]")
    ax[0,1].grid(True,alpha=0.25)

    ax[1,0].plot(d["r_cm"],d["antenna_density"],"o-",label="raw antenna")
    if ANTENNA_ATTACH_MODE!="none":
        ax[1,0].plot(d["r_cm"],d["antenna_density_attached"],"x--",label="attached antenna")
    ax[1,0].plot(d["r_cm"],d["r1_extrapolated"],".-",label="empirical R1 extrapolation")
    ax[1,0].set_title("Absolute radial continuity")
    ax[1,0].set_xlabel("r [cm]"); ax[1,0].set_ylabel("primary-density proxy")
    ax[1,0].legend(fontsize=8); ax[1,0].grid(True,alpha=0.25)

    if len(q):
        for mode in q["lookup_mode"].unique():
            z=q[q["lookup_mode"]==mode].sort_values("ring")
            ax[1,1].plot(z["r_cm"],z["median_density"],"o-",label=mode)
        ax[1,1].set_ylabel("median antenna density")
        ax[1,1].set_title("Global-hist phi lookup convention QA")
        ax[1,1].legend(fontsize=8)
    else:
        ax[1,1].plot(d["r_cm"],d["antenna_over_r1_extrap"],"o-",label="raw/R1")
        ax[1,1].plot(d["r_cm"],d["attached_over_r1_extrap"],"x--",label="attached/R1")
        ax[1,1].axhline(1.0,ls="--",lw=1)
        ax[1,1].legend(fontsize=8)
    ax[1,1].set_xlabel("r [cm]"); ax[1,1].grid(True,alpha=0.25)

    fig.suptitle(f"Antenna diagnostics side {side}")
    path=QA_PLOT_DIR/f"antenna_diagnostics_side{side}.png"
    fig.savefig(path,dpi=160)
    print("saved",path)
    return fig

if not IS_BATCH_CHILD:
    plot_native_stages(*QA_EXAMPLE)
    for _side in range(2):
        plot_radial_diagnostics(_side)
        plot_phi_diagnostics(_side)
        plot_physics_2d(_side)
        plot_variant_r1_comparison(_side)
        plot_antenna_extrapolation(_side)
        plot_antenna_diagnostics(_side)


## 11. Save ROOT and CSV products

The ROOT file keeps the data-like products, all recovery stages for the main pedestal choice, and the smooth-model products separately.

CSV files are also written for:
- scalar run/window summary;
- physical-layer radial profile;
- all recovery-stage layer profiles;
- normalized phi profile.

These lightweight CSVs are what the cross-run analysis uses, so the 57 ROOT files do not need to be reopened for every GEM-current plot.


In [ ]:

def array_to_th2(name,title,a):
    h=root.TH2D(
        name,title,
        len(phi_edges)-1,array("d",phi_edges),
        len(r_edges)-1,array("d",r_edges)
    )
    h.SetDirectory(0)
    for ir in range(a.shape[0]):
        for ip in range(a.shape[1]):
            v=a[ir,ip]
            h.SetBinContent(ip+1,ir+1,float(v) if np.isfinite(v) else 0.0)
    return h

def profile_to_th1(name,title,p,e=None):
    h=root.TH1D(name,title,len(r_edges)-1,array("d",r_edges))
    h.SetDirectory(0)
    for ir,v in enumerate(p):
        if np.isfinite(v):
            h.SetBinContent(ir+1,float(v))
            if e is not None and np.isfinite(e[ir]):
                h.SetBinError(ir+1,float(e[ir]))
    return h

def layer_profile_to_graph(name,title,d,ycol,ecol):
    dd=d[np.isfinite(d[ycol])].sort_values("r_cm")
    g=root.TGraphErrors(len(dd))
    g.SetName(name)
    g.SetTitle(title)
    for i,(_,row) in enumerate(dd.iterrows()):
        y=float(row[ycol])
        e=float(row[ecol]) if np.isfinite(row[ecol]) else 0.0
        g.SetPoint(i,float(row["r_cm"]),y)
        g.SetPointError(i,0.0,e)
    return g

fout=root.TFile(str(OUTPUT_ROOT),"RECREATE")

# ------------------------------------------------------------
# All variants: measured + recovered
# ------------------------------------------------------------
for variant in DC_VARIANTS:
    for mode in ("measured","recovered"):
        for side in range(2):
            item=combined_maps[variant][mode][side]

            for quantity in ("dc","primary"):
                array_to_th2(
                    f"h_{quantity}_density_{mode}_rphi_{variant}_side{side}",
                    f"{quantity} density {mode} {variant} side {side};#phi;r [cm]",
                    item[quantity]
                ).Write()

                p,e,n=profile_cache[(variant,mode,quantity,side)]
                profile_to_th1(
                    f"h_{quantity}_profile_r_{mode}_{variant}_side{side}",
                    f"{quantity} fine-grid robust radial profile {mode} {variant} side {side};r [cm];median over covered #phi",
                    p,e
                ).Write()

                ld=layer_profile_df[
                    (layer_profile_df["variant"]==variant)&
                    (layer_profile_df["mode"]==mode)&
                    (layer_profile_df["side"]==side)
                ]
                layer_profile_to_graph(
                    f"g_{quantity}_physical_layer_profile_{mode}_{variant}_side{side}",
                    f"{quantity} physical-layer profile {mode} {variant} side {side};r [cm];density",
                    ld,f"{quantity}_density",f"{quantity}_err"
                ).Write()

            array_to_th2(
                f"h_coverage_{mode}_{variant}_side{side}",
                f"coverage {mode} {variant} side {side};#phi;r [cm]",
                item["coverage"]
            ).Write()

            if mode=="recovered":
                array_to_th2(
                    f"h_repair_fraction_rphi_{variant}_side{side}",
                    f"repair fraction {variant} side {side};#phi;r [cm]",
                    item["repair_fraction"]
                ).Write()
                array_to_th2(
                    f"h_unstable_fraction_rphi_{variant}_side{side}",
                    f"unstable fraction {variant} side {side};#phi;r [cm]",
                    item["unstable_fraction"]
                ).Write()
                array_to_th2(
                    f"h_antenna_mask_rphi_{variant}_side{side}",
                    f"antenna coverage {variant} side {side};#phi;r [cm]",
                    item["antenna_mask"].astype(float)
                ).Write()

# ------------------------------------------------------------
# Every intermediate recovery step for MAIN_VARIANT
# ------------------------------------------------------------
for stage in ("measured","exposure","structure","stripes","outliers","smoothed"):
    for side in range(2):
        item=main_stage_standard_maps[stage][side]
        for quantity in ("dc","primary"):
            array_to_th2(
                f"h_{quantity}_stage_{stage}_rphi_{MAIN_VARIANT}_side{side}",
                f"{quantity} stage {stage} {MAIN_VARIANT} side {side};#phi;r [cm]",
                item[quantity]
            ).Write()

# ------------------------------------------------------------
# Smooth/equalized model products
# ------------------------------------------------------------
for side in range(2):
    p=physics_products[side]
    array_to_th2(
        f"h_primary_equalized_recovered_rphi_side{side}",
        f"Recovered primary after R2/R3 response equalization side {side};#phi;r [cm]",
        p["equalized_recovered"]
    ).Write()
    array_to_th2(
        f"h_primary_smooth_two_source_rphi_side{side}",
        f"Smooth two-source primary model side {side};#phi;r [cm]",
        p["smooth_model_standard"]
    ).Write()
    array_to_th2(
        f"h_primary_two_source_extrapolated_rphi_side{side}",
        f"Two-source primary model extrapolated inward side {side};#phi;r [cm]",
        p["smooth_model_extrapolated"]
    ).Write()
    array_to_th2(
        f"h_primary_equalized_over_model_rphi_side{side}",
        f"Equalized recovered / smooth model side {side};#phi;r [cm]",
        p["ratio_to_model"]
    ).Write()
    array_to_th2(
        f"h_primary_equalized_minus_model_rphi_side{side}",
        f"Equalized recovered - smooth model side {side};#phi;r [cm]",
        p["residual_to_model"]
    ).Write()

# Backward-friendly aliases remain DATA-LIKE recovered maps.
for side in range(2):
    main=combined_maps[MAIN_VARIANT]["recovered"][side]
    array_to_th2(
        f"h_dc_density_clean_rphi_side{side}",
        f"Recovered DC density side {side};#phi;r [cm]",
        main["dc"]
    ).Write()
    array_to_th2(
        f"h_primary_density_clean_rphi_side{side}",
        f"Recovered primary-density proxy side {side};#phi;r [cm]",
        main["primary"]
    ).Write()

# ------------------------------------------------------------
# Antenna raw/attached products. Raw is NEVER overwritten.
# ------------------------------------------------------------
for side in range(2):
    raw_grid,_,_=antenna_values_to_grid(
        side,antenna_products[MAIN_VARIANT][side]["recovered"]
    )
    array_to_th2(
        f"h_antenna_primary_recovered_raw_rphi_side{side}",
        f"Raw recovered antenna primary proxy side {side};#phi;r [cm]",
        raw_grid
    ).Write()
    array_to_th2(
        f"h_antenna_primary_attached_rphi_side{side}",
        f"Antenna primary after one common response attachment side {side};#phi;r [cm]",
        antenna_attached_grid[side]
    ).Write()
    array_to_th2(
        f"h_primary_density_continuous_rphi_side{side}",
        f"Recovered primary with configured antenna attachment side {side};#phi;r [cm]",
        primary_continuous_map[side]
    ).Write()

# ------------------------------------------------------------
# Run metadata
# ------------------------------------------------------------
hgem=root.TH1D("h_gem_current","R1 GEM current/load;side;load current",2,-0.5,1.5)
hgem.GetXaxis().SetBinLabel(1,"South")
hgem.GetXaxis().SetBinLabel(2,"North")
for s in range(2):
    hgem.SetBinContent(s+1,float(RUN_GEM_CURRENT[s]))
hgem.Write()

hexp=root.TH1D("h_live_frame_slots","Synchronized DC live exposure;side;frame slots",2,-0.5,1.5)
for s in range(2):
    hexp.SetBinContent(s+1,float(LIVE_FRAME_SLOTS[s]))
hexp.Write()

hmodel=root.TH2D(
    "h_two_source_model_summary",
    "two-source model summary;side;parameter",
    2,-0.5,1.5,12,-0.5,11.5
)
for iy,label in enumerate((
    "A_collision","A_beam",
    "alpha_collision","alpha_collision_err",
    "alpha_beam","alpha_beam_err",
    "beam_epsilon","beam_phi0",
    "R2_scale","R3_scale","cost_ndf","n_points"
),1):
    hmodel.GetYaxis().SetBinLabel(iy,label)

for side in range(2):
    f=physics_models[side]
    vals=(
        f["A_collision"],f["A_beam"],
        f["alpha_collision"],f["alpha_collision_err"],
        f["alpha_beam"],f["alpha_beam_err"],
        f["beam_epsilon"],f["beam_phi0"],
        f["module_scale_R2"],f["module_scale_R3"],f["cost_ndf"],f["n"]
    )
    for iy,v in enumerate(vals,1):
        hmodel.SetBinContent(side+1,iy,float(v) if np.isfinite(v) else 0.0)
hmodel.Write()

# Empirical R1 alpha summary
halpha=root.TH2D(
    "h_alpha_summary",
    "R1 alpha summary from physical rows;side;variant index",
    2,-0.5,1.5,len(DC_VARIANTS),-0.5,len(DC_VARIANTS)-0.5
)
for iv,v in enumerate(DC_VARIANTS):
    halpha.GetYaxis().SetBinLabel(iv+1,v)
    for side in range(2):
        sel=alpha_df[
            (alpha_df["variant"]==v)&
            (alpha_df["mode"]=="recovered")&
            (alpha_df["quantity"]=="primary")&
            (alpha_df["side"]==side)
        ]
        if len(sel)==0:
            continue
        row=sel.iloc[0]
        halpha.SetBinContent(side+1,iv+1,float(row["alpha"]))
        if np.isfinite(row["alpha_err"]):
            halpha.SetBinError(
                halpha.GetBin(side+1,iv+1),
                float(row["alpha_err"])
            )
halpha.Write()

# Antenna / empirical-R1 response
har=root.TH2D(
    "h_antenna_over_r1_extrap",
    "Antenna / empirical standard-R1 extrapolation;side;antenna ring",
    2,-0.5,1.5,4,-0.5,3.5
)
for side in range(2):
    for ring in range(4):
        rr=antenna_ratio_df[
            (antenna_ratio_df["side"]==side)&
            (antenna_ratio_df["ring"]==ring)
        ]
        if len(rr):
            har.SetBinContent(
                side+1,ring+1,
                float(rr.iloc[0]["antenna_over_r1_extrap"])
            )
har.Write()

hattach=root.TH1D(
    "h_antenna_attach_factor",
    "Common antenna response attachment factor;side;factor",
    2,-0.5,1.5
)
for side in range(2):
    hattach.SetBinContent(side+1,float(antenna_attach_factor[side]))
hattach.Write()

root.TNamed("dc_main_variant",MAIN_VARIANT).Write()
root.TNamed("dc_defect_variant",DEFECT_VARIANT).Write()
root.TNamed(
    "dc_geometry_definition",
    "v24: standard R1/R2/R3 radial pitch derived separately within each module; "
    "antenna radial pitch derived separately; antenna GLOBAL-HIST lookup uses the configured DC-preprocessing convention while output phi uses canonical IdealPadMap transform."
).Write()
root.TNamed(
    "dc_profile_definition",
    "Empirical alpha fits use physical TPC rows. Fine 0.1-cm grid profiles are display-only."
).Write()
root.TNamed(
    "dc_antenna_lookup_definition",
    f"lookup_mode={ANTENNA_GLOBAL_LOOKUP_MODE}; output phi remains canonical IdealPadMap-style"
).Write()
root.TNamed(
    "dc_antenna_attach_definition",
    f"attach_mode={ANTENNA_ATTACH_MODE}; one common multiplicative response factor per side; raw antenna always saved"
).Write()

root.TNamed(
    "dc_smooth_model_definition",
    f"rho=A_collision*(Rref/r)^alpha_collision + A_beam*(Rref/r)^alpha_beam*[1+epsilon*cos(phi-phi0)]; "
    f"alpha_collision fit in {COLLISION_ALPHA_BOUNDS} from {COLLISION_ALPHA_INITIAL}; "
    f"alpha_beam fit in {BEAM_ALPHA_BOUNDS} from {BEAM_ALPHA_INITIAL}; "
    "R2/R3 detector-response scales are fitted as explicit nuisance parameters."
).Write()

fout.Close()

# ------------------------------------------------------------
# CSV products used for cross-run analysis
# ------------------------------------------------------------
physics_layer_df.to_csv(OUTPUT_LAYER_PROFILE,index=False)
stage_layer_profile_df.to_csv(OUTPUT_STAGE_LAYER_PROFILE,index=False)
phi_profile_df.to_csv(OUTPUT_PHI_PROFILE,index=False)

summary_rows=[]
for side in range(2):
    model=physics_models[side]
    for variant in DC_VARIANTS:
        sel=alpha_df[
            (alpha_df["variant"]==variant)&
            (alpha_df["mode"]=="recovered")&
            (alpha_df["quantity"]=="primary")&
            (alpha_df["side"]==side)
        ]
        if len(sel)==0:
            continue
        ar=sel.iloc[0]

        # Stable DC metric from the 48 physical standard rows.
        ld=layer_profile_df[
            (layer_profile_df["variant"]==variant)&
            (layer_profile_df["mode"]=="recovered")&
            (layer_profile_df["side"]==side)
        ]
        dc_metric=float(np.nanmedian(ld["dc_density"])) if len(ld) else np.nan

        row={
            "run":RUN_NUMBER,
            "window":WINDOW_NUMBER,
            "side":side,
            "variant":variant,
            "gem_current":RUN_GEM_CURRENT[side],
            "live_frame_slots":LIVE_FRAME_SLOTS[side],
            "dc_density_metric":dc_metric,
            "alpha_primary":float(ar["alpha"]),
            "alpha_primary_err":float(ar["alpha_err"]),
        }

        if variant==MAIN_VARIANT:
            row.update({
                "model_A_collision":model["A_collision"],
                "model_A_beam":model["A_beam"],
                "model_alpha_collision":model["alpha_collision"],
                "model_alpha_collision_err":model["alpha_collision_err"],
                "model_alpha_beam":model["alpha_beam"],
                "model_alpha_beam_err":model["alpha_beam_err"],
                "model_alpha_collision_at_bound":model.get("collision_alpha_at_bound",False),
                "model_alpha_beam_at_bound":model.get("beam_alpha_at_bound",False),
                "model_beam_epsilon":model["beam_epsilon"],
                "model_beam_phi0":model["beam_phi0"],
                "model_R2_scale":model["module_scale_R2"],
                "model_R3_scale":model["module_scale_R3"],
                "model_cost_ndf":model["cost_ndf"],
            })
        else:
            row.update({
                "model_A_collision":np.nan,
                "model_A_beam":np.nan,
                "model_alpha_collision":np.nan,
                "model_alpha_collision_err":np.nan,
                "model_alpha_beam":np.nan,
                "model_alpha_beam_err":np.nan,
                "model_alpha_collision_at_bound":np.nan,
                "model_alpha_beam_at_bound":np.nan,
                "model_beam_epsilon":np.nan,
                "model_beam_phi0":np.nan,
                "model_R2_scale":np.nan,
                "model_R3_scale":np.nan,
                "model_cost_ndf":np.nan,
            })

        for ring in range(4):
            rr=antenna_ratio_df[
                (antenna_ratio_df["side"]==side)&
                (antenna_ratio_df["ring"]==ring)
            ]
            row[f"antenna_ratio_A{ring}"]=(
                float(rr.iloc[0]["antenna_over_r1_extrap"])
                if len(rr) else np.nan
            )
            row[f"antenna_attached_ratio_A{ring}"]=(
                float(rr.iloc[0]["attached_over_r1_extrap"])
                if len(rr) else np.nan
            )
        row["antenna_attach_factor"]=float(antenna_attach_factor[side])
        row["antenna_lookup_mode"]=ANTENNA_GLOBAL_LOOKUP_MODE
        row["antenna_attach_mode"]=ANTENNA_ATTACH_MODE

        summary_rows.append(row)

summary_df=pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_SUMMARY,index=False)

print("Wrote",OUTPUT_ROOT)
print("Wrote",OUTPUT_SUMMARY)
print("Wrote",OUTPUT_LAYER_PROFILE)
print("Wrote",OUTPUT_STAGE_LAYER_PROFILE)
print("Wrote",OUTPUT_PHI_PROFILE)

## 12. Parallel processing of all Takao windows

The local input is now discovered with:

```text
/media/yoren/T7_Shield/sPHENIX/tracking/dc/dc_run*.root
```

This is expected to find the 57 early/middle/late window files.

Each file is a separate batch item. Output tags include **both run and window**, for example:

```text
run80858_win00
run80858_win01
run80858_win02
```

so parallel workers cannot overwrite one another.

`ThreadPoolExecutor` is only orchestrating independent `jupyter nbconvert` subprocesses; the heavy ROOT work remains isolated per process. `BATCH_MAX_WORKERS=6` is a conservative default for the external SSD and is configurable.


In [ ]:

def discover_batch_inputs(pattern=BATCH_INPUT_GLOB):
    # glob.glob supports both absolute and relative patterns.
    files=sorted(glob.glob(str(pattern)))
    return [str(Path(p).resolve()) for p in files]

def describe_batch_inputs(input_files):
    rows=[]
    for p in input_files:
        rows.append({
            "file":p,
            "run":infer_run(p),
            "window":infer_window(p),
        })
    d=pd.DataFrame(rows)
    if len(d):
        print(f"Found {len(d)} ROOT files, {d['run'].nunique()} runs")
        print(d.groupby("run")["window"].apply(list).to_string())
    return d

def run_batch(input_files=None, notebook=NOTEBOOK_FILE, max_workers=BATCH_MAX_WORKERS):
    if IS_BATCH_CHILD:
        print("Batch child: production-only execution")
        return []

    if input_files is None:
        input_files=discover_batch_inputs()

    if not input_files:
        print("No batch inputs found with:",BATCH_INPUT_GLOB)
        return []

    describe_batch_inputs(input_files)

    notebook=Path(notebook).resolve()
    if not notebook.exists():
        raise FileNotFoundError(
            f"Notebook not found: {notebook}. "
            "Set TPC_DC_NOTEBOOK if the filename/path differs."
        )

    print(f"Launching {len(input_files)} files with max_workers={max_workers}")

    def run_one(infile):
        infile=Path(infile).resolve()
        run=infer_run(infile)
        window=infer_window(infile)

        if run>0 and window>=0:
            tag=f"run{run}_win{window:02d}"
        elif run>0:
            tag=f"run{run}"
        else:
            tag=infile.stem

        env=os.environ.copy()
        env["TPC_DC_INPUT"]=str(infile)
        env["TPC_DC_TAG"]=tag
        env["TPC_DC_BATCH_CHILD"]="1"
        env["TPC_DC_NOTEBOOK"]=str(notebook)

        executed=(OUTPUT_DIR/f"executed_v24_{tag}.ipynb").resolve()
        cmd=[
            "jupyter","nbconvert","--to","notebook","--execute",str(notebook),
            "--output",executed.name,
            "--output-dir",str(executed.parent),
            "--ExecutePreprocessor.timeout=-1",
        ]

        print(f"[START] {tag}: {infile}")
        result=subprocess.run(cmd,env=env,text=True,capture_output=True)
        if result.returncode:
            raise RuntimeError(
                f"{tag} failed (exit {result.returncode})\n"
                f"STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
            )
        print(f"[DONE ] {tag}")
        return {"run":run,"window":window,"tag":tag,"file":str(infile)}

    produced=[]
    failures=[]
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures={pool.submit(run_one,f):f for f in input_files}
        for future in as_completed(futures):
            try:
                produced.append(future.result())
            except Exception as exc:
                failures.append((futures[future],str(exc)))
                print("[FAIL]",futures[future],exc)

    produced=sorted(produced,key=lambda x:(x["run"],x["window"]))
    print(f"Finished {len(produced)}/{len(input_files)} files")
    if failures:
        print(f"Failures: {len(failures)}")
        for f,msg in failures:
            print(" ",f,":",msg[:500])
    else:
        print("All batch files finished successfully.")

    return produced

# First inspect what will run:
# batch_files = discover_batch_inputs()
# describe_batch_inputs(batch_files)
#
# Then launch all 57 files:
# produced_files = run_batch(batch_files)


## 13. Cross-run GEM-current and response study

After the 57 first-pass files are processed, this section works only from their small CSV summaries.

It produces:

- recovered DC density vs GEM current using every window plus run-averaged points;
- empirical R1 alpha vs GEM current;
- fitted R2/R3 response scale vs GEM current;
- collision and beam-background amplitudes vs GEM current;
- beam \(n=1\) amplitude and preferred direction vs GEM current;
- antenna/R1 response vs GEM current;
- run/reference DC-density ratios;
- a cross-run static layer-response diagnostic.

The layer-response correction removes only repeatable high-frequency layer structure after the broad radial model and R2/R3 module scales have been removed. It is written to `TPC_DC_layer_response_v24.csv` but **is not applied automatically on the first pass**.


In [ ]:

def load_batch_summaries():
    paths=sorted(OUTPUT_DIR.glob("TPC_DC_summary_v24_run*_win*.csv"))
    frames=[]
    for p in paths:
        try:
            frames.append(pd.read_csv(p))
        except Exception as exc:
            print("Skipping",p,exc)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames,ignore_index=True).sort_values(
        ["run","window","side","variant"]
    )

def load_batch_layer_profiles():
    paths=sorted(OUTPUT_DIR.glob("TPC_DC_layer_profile_v24_run*_win*.csv"))
    frames=[]
    for p in paths:
        try:
            frames.append(pd.read_csv(p))
        except Exception as exc:
            print("Skipping",p,exc)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames,ignore_index=True).sort_values(
        ["run","window","side","global_layer"]
    )

def fit_line(x,y):
    x=np.asarray(x,float)
    y=np.asarray(y,float)
    good=np.isfinite(x)&np.isfinite(y)
    if np.count_nonzero(good)<2:
        return np.nan,np.nan
    p=np.polyfit(x[good],y[good],1)
    return float(p[0]),float(p[1])

def add_linear_trend(ax,x,y,label):
    x=np.asarray(x,float)
    y=np.asarray(y,float)
    good=np.isfinite(x)&np.isfinite(y)
    if np.count_nonzero(good)<2:
        return
    slope,intercept=fit_line(x[good],y[good])
    xx=np.linspace(np.min(x[good]),np.max(x[good]),100)
    ax.plot(xx,intercept+slope*xx,"--",label=f"{label} slope={slope:.4g}")

def run_average_table(d):
    numeric=[
        "gem_current","dc_density_metric","alpha_primary",
        "model_A_collision","model_A_beam",
        "model_alpha_collision","model_alpha_collision_err",
        "model_alpha_beam","model_alpha_beam_err",
        "model_beam_epsilon",
        "model_beam_phi0","model_R2_scale","model_R3_scale",
        "model_cost_ndf",
        "antenna_ratio_A0","antenna_ratio_A1","antenna_ratio_A2","antenna_ratio_A3",
        "antenna_attach_factor",
    ]
    agg={}
    for c in numeric:
        if c in d.columns:
            agg[c]=["mean","std","count"]
    if not agg:
        return pd.DataFrame()
    out=d.groupby(["run","side"]).agg(agg)
    out.columns=["_".join(x) for x in out.columns]
    return out.reset_index()

def run_comparisons(variant=MAIN_VARIANT):
    df=load_batch_summaries()
    if df.empty:
        print("No v24 batch summaries found in",OUTPUT_DIR)
        return None

    d=df[df["variant"]==variant].copy()
    outcsv=OUTPUT_DIR/f"TPC_DC_crossrun_v24_{variant}.csv"
    d.to_csv(outcsv,index=False)
    print("Wrote",outcsv)
    print(
        f"Cross-run sample: {len(d)//2} files, "
        f"{d['run'].nunique()} runs, {sorted(d['window'].dropna().unique())} windows"
    )

    ravg=run_average_table(d)
    ravg.to_csv(
        OUTPUT_DIR/f"TPC_DC_crossrun_runavg_v24_{variant}.csv",
        index=False
    )

    # --------------------------------------------------------
    # 1) Stable DC/ADC-density proxy vs GEM current
    # --------------------------------------------------------
    fig,ax=plt.subplots(figsize=(8.5,6.2))
    for side in range(2):
        s=d[d["side"]==side]
        ax.plot(
            s["gem_current"],s["dc_density_metric"],"o",
            label=f"side {side}: all windows"
        )
        add_linear_trend(
            ax,s["gem_current"],s["dc_density_metric"],f"side {side}"
        )

        a=ravg[ravg["side"]==side]
        if len(a):
            ax.errorbar(
                a["gem_current_mean"],a["dc_density_metric_mean"],
                xerr=a["gem_current_std"],yerr=a["dc_density_metric_std"],
                fmt="s",ms=7,label=f"side {side}: run average"
            )

    ax.set_xlabel("R1 GEM current/load")
    ax.set_ylabel("stable recovered DC density proxy")
    ax.set_title(f"DC density vs GEM current ({variant})")
    ax.grid(True,alpha=0.25)
    ax.legend(fontsize=8)
    p=OUTPUT_DIR/f"dc_density_vs_gem_current_v24_{variant}.png"
    fig.savefig(p,dpi=170,bbox_inches="tight")
    print("saved",p)

    # --------------------------------------------------------
    # 2) Empirical R1 alpha vs GEM current
    # --------------------------------------------------------
    fig,ax=plt.subplots(figsize=(8.5,6.2))
    for side in range(2):
        s=d[d["side"]==side]
        ax.errorbar(
            s["gem_current"],s["alpha_primary"],
            yerr=s["alpha_primary_err"],
            fmt="o",label=f"side {side}"
        )
    ax.axhspan(COLLISION_ALPHA_BOUNDS[0],COLLISION_ALPHA_BOUNDS[1],alpha=0.08,
               label=f"collision fit range {COLLISION_ALPHA_BOUNDS}")
    ax.axhspan(BEAM_ALPHA_BOUNDS[0],BEAM_ALPHA_BOUNDS[1],alpha=0.05,
               label=f"beam fit range {BEAM_ALPHA_BOUNDS}")
    ax.axhline(COLLISION_ALPHA_INITIAL,ls="--",lw=1,label=f"collision start={COLLISION_ALPHA_INITIAL}")
    ax.axhline(BEAM_ALPHA_INITIAL,ls=":",lw=1,label=f"beam start={BEAM_ALPHA_INITIAL}")
    ax.set_xlabel("R1 GEM current/load")
    ax.set_ylabel("empirical standard-R1 alpha")
    ax.set_title(f"R1 alpha vs GEM current ({variant})")
    ax.grid(True,alpha=0.25)
    ax.legend(fontsize=8)
    p=OUTPUT_DIR/f"alpha_vs_gem_current_v24_{variant}.png"
    fig.savefig(p,dpi=170,bbox_inches="tight")
    print("saved",p)

    # --------------------------------------------------------
    # 3) Fitted two-source alpha values vs GEM current
    # --------------------------------------------------------
    fig,ax=plt.subplots(figsize=(8.5,6.2))
    for side in range(2):
        q=d[d["side"]==side]
        ax.errorbar(q["gem_current"],q["model_alpha_collision"],
                    yerr=q["model_alpha_collision_err"],fmt="o",
                    label=f"side {side} collision")
        ax.errorbar(q["gem_current"],q["model_alpha_beam"],
                    yerr=q["model_alpha_beam_err"],fmt="s",
                    label=f"side {side} beam")
    ax.axhspan(COLLISION_ALPHA_BOUNDS[0],COLLISION_ALPHA_BOUNDS[1],alpha=0.08)
    ax.axhspan(BEAM_ALPHA_BOUNDS[0],BEAM_ALPHA_BOUNDS[1],alpha=0.05)
    ax.axhline(COLLISION_ALPHA_INITIAL,ls="--",lw=1)
    ax.axhline(BEAM_ALPHA_INITIAL,ls=":",lw=1)
    ax.set_xlabel("R1 GEM current/load")
    ax.set_ylabel("fitted radial exponent")
    ax.set_title(f"Two-source fitted alpha values vs GEM current ({variant})")
    ax.grid(True,alpha=0.25)
    ax.legend(fontsize=8,ncol=2)
    p=OUTPUT_DIR/f"model_alphas_vs_gem_current_v24_{variant}.png"
    fig.savefig(p,dpi=170,bbox_inches="tight")
    print("saved",p)

    # --------------------------------------------------------
    # 4) R2/R3 response scale vs GEM current
    #    This is the main diagnostic for the radial discontinuities.
    # --------------------------------------------------------
    fig,ax=plt.subplots(figsize=(8.5,6.2))
    for side in range(2):
        s=d[d["side"]==side]
        ax.plot(s["gem_current"],s["model_R2_scale"],"o",
                label=f"side {side} R2/R1")
        ax.plot(s["gem_current"],s["model_R3_scale"],"s",
                label=f"side {side} R3/R1")
    ax.axhline(1.0,ls="--",lw=1)
    ax.set_xlabel("R1 GEM current/load")
    ax.set_ylabel("fitted module response scale")
    ax.set_title("Module response scales vs GEM current")
    ax.grid(True,alpha=0.25)
    ax.legend(fontsize=8,ncol=2)
    p=OUTPUT_DIR/f"module_scales_vs_gem_current_v24_{variant}.png"
    fig.savefig(p,dpi=170,bbox_inches="tight")
    print("saved",p)

    # --------------------------------------------------------
    # 4) Smooth model component amplitudes vs GEM current
    # --------------------------------------------------------
    fig,ax=plt.subplots(figsize=(8.5,6.2))
    for side in range(2):
        s=d[d["side"]==side]
        ax.plot(s["gem_current"],s["model_A_collision"],"o",
                label=f"side {side} collision")
        ax.plot(s["gem_current"],s["model_A_beam"],"s",
                label=f"side {side} beam")
    ax.set_xlabel("R1 GEM current/load")
    ax.set_ylabel("two-source amplitude at Rref")
    ax.set_title("Smooth radial-source amplitudes vs GEM current")
    ax.grid(True,alpha=0.25)
    ax.legend(fontsize=8,ncol=2)
    p=OUTPUT_DIR/f"source_amplitudes_vs_gem_current_v24_{variant}.png"
    fig.savefig(p,dpi=170,bbox_inches="tight")
    print("saved",p)

    # --------------------------------------------------------
    # 5) Beam n=1 phi amplitude and direction vs GEM current
    # --------------------------------------------------------
    fig,ax=plt.subplots(1,2,figsize=(14,5.8),constrained_layout=True)
    for side in range(2):
        s=d[d["side"]==side]
        ax[0].plot(s["gem_current"],s["model_beam_epsilon"],"o",
                   label=f"side {side}")
        ax[1].plot(s["gem_current"],s["model_beam_phi0"],"o",
                   label=f"side {side}")
    ax[0].axhline(0.0,ls="--",lw=1)
    ax[0].set_xlabel("R1 GEM current/load")
    ax[0].set_ylabel("beam n=1 epsilon")
    ax[0].set_title("Beam-background phi modulation")
    ax[0].grid(True,alpha=0.25)
    ax[0].legend()

    ax[1].axhline(0.0,ls="--",lw=1)
    ax[1].set_xlabel("R1 GEM current/load")
    ax[1].set_ylabel("preferred direction phi0 [rad]")
    ax[1].set_title("Beam-background preferred direction")
    ax[1].grid(True,alpha=0.25)
    ax[1].legend()

    p=OUTPUT_DIR/f"beam_phi_vs_gem_current_v24_{variant}.png"
    fig.savefig(p,dpi=170,bbox_inches="tight")
    print("saved",p)

    # --------------------------------------------------------
    # 6) Antenna / empirical-R1 response vs GEM current
    # --------------------------------------------------------
    fig,ax=plt.subplots(figsize=(9,6.2))
    for side in range(2):
        s=d[d["side"]==side]
        for ring in range(4):
            col=f"antenna_ratio_A{ring}"
            ax.plot(
                s["gem_current"],s[col],"o",
                label=f"s{side} A{ring}"
            )
    ax.axhline(1.0,ls="--",lw=1)
    ax.set_xlabel("R1 GEM current/load")
    ax.set_ylabel("antenna / empirical standard-R1 extrapolation")
    ax.set_title(f"Antenna response vs GEM current ({variant})")
    ax.grid(True,alpha=0.25)
    ax.legend(ncol=2,fontsize=8)
    p=OUTPUT_DIR/f"antenna_ratio_vs_gem_current_v24_{variant}.png"
    fig.savefig(p,dpi=170,bbox_inches="tight")
    print("saved",p)

    # --------------------------------------------------------
    # 7) DC density ratio to the reference run
    # --------------------------------------------------------
    fig,ax=plt.subplots(figsize=(9,6.2))
    for side in range(2):
        s=d[d["side"]==side].copy()
        ref=s[s["run"]==REFERENCE_RUN]
        if ref.empty:
            print(f"Reference run {REFERENCE_RUN} missing for side {side}")
            continue
        refv=float(np.nanmedian(ref["dc_density_metric"]))
        ratio=s["dc_density_metric"]/refv
        ax.plot(s["run"]+0.05*s["window"],ratio,"o",label=f"side {side}")
    ax.axhline(1.0,ls="--",lw=1)
    ax.set_xlabel("run + small window offset")
    ax.set_ylabel(f"DC density / run {REFERENCE_RUN}")
    ax.set_title(f"Run/reference DC-density ratio ({variant})")
    ax.grid(True,alpha=0.25)
    ax.legend()
    p=OUTPUT_DIR/f"dc_density_ratio_to_{REFERENCE_RUN}_v24_{variant}.png"
    fig.savefig(p,dpi=170,bbox_inches="tight")
    print("saved",p)

    return d

def derive_crossrun_layer_response():
    """
    Derive a STATIC per-layer response correction from all windows.

    We first remove the fitted R2/R3 module scale and broad two-source radial
    trend for every file. The remaining median layer ratio is then divided by
    a smooth within-module baseline. Therefore this correction targets stable
    layer-to-layer structure, not the broad physical radial dependence.
    """
    d=load_batch_layer_profiles()
    if d.empty:
        print("No v24 layer-profile CSV files found.")
        return None

    rows=[]
    for side in range(2):
        for module in range(3):
            s=d[(d["side"]==side)&(d["module"]==module)].copy()
            if s.empty:
                continue

            # One robust cross-file ratio per physical layer.
            layer=[]
            for gl in sorted(s["global_layer"].unique()):
                q=s[s["global_layer"]==gl]
                vals=q["ratio_after_equalization"].to_numpy(float)
                vals=vals[np.isfinite(vals)&(vals>0)]
                if not len(vals):
                    continue
                med=np.median(vals)
                mad=1.4826*np.median(np.abs(vals-med))
                first=q.iloc[0]
                layer.append({
                    "side":side,
                    "module":module,
                    "local_layer":int(first["local_layer"]),
                    "global_layer":int(gl),
                    "r_cm":float(first["r_cm"]),
                    "median_ratio":float(med),
                    "scatter":float(mad),
                    "n_files":int(len(vals)),
                })

            if not layer:
                continue

            ld=pd.DataFrame(layer).sort_values("local_layer")
            v=ld["median_ratio"].to_numpy(float)

            # Smooth baseline within each module. Only the high-frequency,
            # repeatable layer structure is turned into a calibration factor.
            smooth=ndimage.gaussian_filter1d(v,sigma=2.0,mode="nearest")
            correction=np.divide(
                v,smooth,
                out=np.ones_like(v),
                where=np.isfinite(smooth)&(smooth>0)
            )
            correction/=np.nanmedian(correction)

            ld["smooth_baseline"]=smooth
            ld["correction"]=correction
            rows.extend(ld.to_dict("records"))

    corr=pd.DataFrame(rows).sort_values(["side","global_layer"])
    LAYER_RESPONSE_FILE.parent.mkdir(parents=True,exist_ok=True)
    corr.to_csv(LAYER_RESPONSE_FILE,index=False)
    print("Wrote",LAYER_RESPONSE_FILE)

    fig,ax=plt.subplots(2,1,figsize=(11,8),constrained_layout=True)
    for side in range(2):
        s=corr[corr["side"]==side]
        ax[0].errorbar(
            s["r_cm"],s["median_ratio"],yerr=s["scatter"],
            fmt=".",label=f"side {side}"
        )
        ax[1].plot(
            s["r_cm"],s["correction"],".-",
            label=f"side {side}"
        )
    ax[0].axhline(1.0,ls="--",lw=1)
    ax[0].set_xlabel("r [cm]")
    ax[0].set_ylabel("cross-file layer/model ratio")
    ax[0].set_title("Persistent physical-layer response before high-frequency correction")
    ax[0].grid(True,alpha=0.25)
    ax[0].legend()

    ax[1].axhline(1.0,ls="--",lw=1)
    ax[1].set_xlabel("r [cm]")
    ax[1].set_ylabel("recommended layer response correction")
    ax[1].set_title("Static layer-to-layer correction (median=1 within each module)")
    ax[1].grid(True,alpha=0.25)
    ax[1].legend()

    p=OUTPUT_DIR/"crossrun_layer_response_v24.png"
    fig.savefig(p,dpi=170,bbox_inches="tight")
    print("saved",p)

    return corr

def run_all_dc_files(max_workers=BATCH_MAX_WORKERS):
    """
    One-call first pass:
      1. discover all 57 window files;
      2. process them in parallel;
      3. make GEM-current dependence plots;
      4. derive the static layer-response correction.

    After inspecting the correction, a second pass can apply it by setting:
      TPC_DC_APPLY_LAYER_CORRECTION=1
    """
    files=discover_batch_inputs()
    describe_batch_inputs(files)
    produced=run_batch(files,max_workers=max_workers)
    comparison=run_comparisons(MAIN_VARIANT)
    correction=derive_crossrun_layer_response()
    return produced,comparison,correction

# Recommended first-pass command after validating one file:
# produced_files, comparison_df, layer_response_df = run_all_dc_files()
#
# To compare positive clipping as a cross-check:
# comparison_df_positive = run_comparisons("ped60_positive")


## Recommended workflow

For one file first:

1. Inspect `radial_diagnostics_side0/1`. The most important question is whether the R3 irregularity is already present in **DC** or appears mainly after the gain correction.
2. Inspect the fitted `R2 scale` and `R3 scale`. A stable scale different from 1 across many windows points to detector response rather than radial physics.
3. Inspect `phi_diagnostics_side0/1`. The smooth model intentionally keeps only the \(n=1\) beam-background harmonic.
4. Compare `measured → structure → stripes → outliers → smoothed`; no recovery step is hidden.
5. Inspect `antenna_diagnostics_side0/1` first. Raw antenna values are preserved; the optional common attachment factor is a response QA/continuity tool, not a per-ring physics correction.

Then process all local files:

```python
batch_files = discover_batch_inputs()
describe_batch_inputs(batch_files)
produced_files = run_batch(batch_files)
comparison_df = run_comparisons("ped60_signed")
layer_response_df = derive_crossrun_layer_response()
```

or simply:

```python
produced_files, comparison_df, layer_response_df = run_all_dc_files()
```

After reviewing `crossrun_layer_response_v24.png`, a second pass can apply the static layer correction:

```bash
export TPC_DC_APPLY_LAYER_CORRECTION=1
export TPC_DC_LAYER_RESPONSE_FILE=output_dc/TPC_DC_layer_response_v24.csv
```

The smooth two-source map and the detector-recovered map remain separate even in that second pass.
